# 🛟 Bayview IT Assistant - Multi-Agent IT Support (LangGraph + Gemini)
**BUS 118s-01 Capstone.** Four cooperating agents resolve IT requests:

| Agent | Role |
|---|---|
| 🧭 Intake | Redacts PII, classifies the request, extracts entities (Gemini structured output) |
| 📚 Knowledge | RAG: Gemini embeddings → Chroma vector DB → grounded answer with citations |
| ⚙️ Workflow | Automations: unlock account, password reset, service status, log analysis |
| 🙋 Escalation | Writes a technician handoff and opens a routed ticket (optional GitHub issue) |

Run all cells top to bottom (Runtime → Run all).

## 1 · Install dependencies

In [ ]:
!pip install -q langgraph google-genai chromadb gradio "mcp[cli]" pydantic python-dotenv numpy matplotlib

## 🔑 Gemini API key
This notebook reads your key from a **Colab secret named `GEMINI_API_KEY`**:
1. Click the 🔑 **Secrets** icon in the left sidebar.
2. Add a secret named `GEMINI_API_KEY` with your key from https://aistudio.google.com/apikey
3. Turn on **Notebook access** for it.

The key is never printed. Without it, the system runs in clearly-labelled offline mode (rule-based baseline).

In [ ]:
import os
from google.colab import userdata
try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ GEMINI_API_KEY loaded from Colab secrets")
except Exception as e:
    print("⚠️ GEMINI_API_KEY secret not available - running in offline mode.", type(e).__name__)

# Optional tuning (see helpdesk/config.py):
os.environ.setdefault("GEMINI_MODEL", "gemini-flash-latest")      # current stable Flash
os.environ.setdefault("GEMINI_MIN_INTERVAL", "4")                  # seconds between calls; keeps free tier under RPM limits
# os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")        # optional: mirror escalations to GitHub Issues
# os.environ["GITHUB_REPO"] = "your-user/your-repo"

## 2 · Project files
The cells below write the project source into `/content/Capstone_ITSupport_Agents`. Each module has a docstring explaining its role.

In [ ]:
%mkdir -p /content/Capstone_ITSupport_Agents/helpdesk /content/Capstone_ITSupport_Agents/scripts /content/Capstone_ITSupport_Agents/eval /content/Capstone_ITSupport_Agents/data/kb /content/Capstone_ITSupport_Agents/data/logs
%cd /content/Capstone_ITSupport_Agents

### Data: 12 knowledge-base articles, mock user directory, service status, system logs and evaluation scenarios

In [ ]:
# Seed data (knowledge base, directory, services, logs, scenarios)
import json, pathlib
DATA = json.loads(r'''{
 "data/kb/KB-001-password-reset.md": "---\nid: KB-001\ntitle: How to reset your network password\ncategory: password_reset\nupdated: 2026-08-14\n---\n# How to reset your network password\n\nYour network password is used for Windows sign-in, email, VPN and the SSO portal. Passwords expire every 90 days.\n\n## Self-service reset (recommended)\n1. Go to the self-service portal at https://passwordreset.bayview.example and click **Forgot password**.\n2. Enter your username (the part of your email before the @).\n3. Approve the MFA prompt on your registered phone (Microsoft Authenticator push or SMS code).\n4. Choose a new password and sign out of all devices, then sign back in.\n\n## Password rules\n- At least 14 characters; passphrases such as `harbor-lantern-quiet-47` are encouraged.\n- Cannot reuse any of your last 10 passwords.\n- Must not contain your name or username.\n\n## After resetting\n- Update the saved password on your phone's mail app and on the Wi-Fi profile `Bayview-Corp`, or your account may lock again from repeated old-password attempts.\n- VPN uses the same password; reconnect after the change.\n\n## When to contact the Service Desk\n- You are not enrolled in MFA and cannot complete the self-service reset (see KB-003).\n- You receive \"account locked\" after resetting (see KB-002).\nThe Service Desk will never ask you for your password.\n",
 "data/kb/KB-002-account-lockout.md": "---\nid: KB-002\ntitle: Account locked out\ncategory: account_lockout\nupdated: 2026-07-02\n---\n# Account locked out\n\nAccounts lock automatically after **5 failed sign-in attempts within 15 minutes**. A lockout lasts 30 minutes and then clears itself.\n\n## Common causes\n- A phone or tablet still using an old password for email (most common after a password change).\n- A mapped network drive or saved credential in Windows Credential Manager with the old password.\n- Typing the password with Caps Lock on.\n- Someone else trying to sign in to your account. Repeated lockouts with no obvious cause should be reported as a security concern (see KB-011).\n\n## Unlocking\n- Wait 30 minutes, or request an unlock through the IT Assistant or Service Desk. Identity is verified with your MFA device before unlocking.\n- Then remove stale saved passwords: Control Panel > Credential Manager > Windows Credentials, delete entries for `bayview` servers.\n\n## Prevention\n- Update your password on every device right after a reset.\n- Use the Company Portal app on mobile instead of manually configured mail profiles.\n",
 "data/kb/KB-003-mfa-setup.md": "---\nid: KB-003\ntitle: Setting up or replacing your MFA device\ncategory: account_access\nupdated: 2026-05-20\n---\n# Setting up or replacing your MFA device\n\nMulti-factor authentication (MFA) is required for email, VPN, SSO and self-service password reset.\n\n## First-time setup\n1. Install **Microsoft Authenticator** from the App Store or Google Play.\n2. On a computer, sign in to https://mysignins.bayview.example and choose **Security info > Add method > Authenticator app**.\n3. Scan the QR code with the app and approve the test notification.\n4. Add a phone number as a backup method.\n\n## New or lost phone\n- If you still have the old phone: add the new phone as a method first, then remove the old one.\n- If the old phone is lost or wiped: you cannot reset MFA yourself. Contact the Service Desk; an agent must verify your identity (employee ID plus manager confirmation) before clearing MFA methods. Lost company phones must also be reported as a security incident (KB-011).\n\n## Tips\n- Do not approve MFA prompts you did not start. Unexpected prompts mean someone may know your password: reset it immediately (KB-001) and report it (KB-011).\n",
 "data/kb/KB-004-vpn-troubleshooting.md": "---\nid: KB-004\ntitle: VPN (GlobalConnect) connection problems\ncategory: vpn\nupdated: 2026-09-03\n---\n# VPN (GlobalConnect) connection problems\n\nBayview uses the GlobalConnect VPN client. Portal address: `vpn.bayview.example`.\n\n## Quick checks\n1. Confirm your internet connection works without VPN (open any public website).\n2. Check the IT status page or ask the IT Assistant whether the VPN service has a known outage.\n3. Make sure the portal address is exactly `vpn.bayview.example`.\n\n## Error messages\n- **\"Authentication failed\"**: your password may have expired or changed. Reset it (KB-001) and reconnect. Repeated failures can lock your account (KB-002).\n- **\"Gateway not responding\" / stuck on \"Connecting\"**: quit GlobalConnect fully (right-click tray icon > Quit), then reopen it. If you are on hotel or airport Wi-Fi, accept the captive portal page in a browser first.\n- **\"Certificate error\"**: your laptop's device certificate may have expired. Restart the laptop while connected to the office network or run **Company Portal > Sync**. If it persists, contact the Service Desk.\n- **Connected but cannot reach internal sites**: disconnect, run `ipconfig /flushdns` in Command Prompt, reconnect.\n\n## Still not working?\nCollect the GlobalConnect logs (Settings > Troubleshooting > Collect Logs) and contact the Service Desk with the time of the failure.\n",
 "data/kb/KB-005-email-outlook.md": "---\nid: KB-005\ntitle: Outlook and email issues\ncategory: email\nupdated: 2026-06-11\n---\n# Outlook and email issues\n\n## Outlook will not open or keeps asking for a password\n1. Close Outlook completely, then open it again.\n2. If prompted repeatedly for a password, your password may have changed: sign out of Office (File > Office Account > Sign out), restart, and sign in again.\n3. Start Outlook in safe mode (`outlook.exe /safe`) to rule out a broken add-in. Disable add-ins under File > Options > Add-ins if safe mode works.\n\n## Emails not arriving\n- Check the **Junk Email** and **Other** (Focused Inbox) folders.\n- Check whether your mailbox is full: the default quota is 50 GB. Archive or delete large items.\n- External senders may be blocked by the spam filter; ask IT to review quarantine.\n\n## Shared mailboxes and distribution lists\n- Access to shared mailboxes must be approved by the mailbox owner; submit an access request (KB-010).\n\n## Mobile mail\n- Use the Outlook mobile app. Native mail apps are not supported and are a common cause of account lockouts (KB-002).\n",
 "data/kb/KB-006-printer.md": "---\nid: KB-006\ntitle: Printer and scanning problems\ncategory: printer\nupdated: 2026-04-28\n---\n# Printer and scanning problems\n\nAll office printers use secure release printing (\"Follow-Me Print\"). You print to the queue `BAYVIEW-SECURE` and release the job at any printer by tapping your badge.\n\n## Job does not print\n1. Confirm you selected the `BAYVIEW-SECURE` printer, not a printer from a previous office.\n2. Tap your badge on the printer panel and look for the job in the list. Jobs are held for 24 hours.\n3. If your badge is not recognized, register it once: tap the badge, sign in with your username and password on the panel.\n\n## Adding the printer\n- Windows: Settings > Bluetooth & devices > Printers & scanners > Add device > `BAYVIEW-SECURE`.\n- If it does not appear, connect to VPN or the office network and run Company Portal > Sync.\n\n## Paper jams and toner\n- Follow the on-screen diagram. For toner or repeated jams, report the printer's asset tag (sticker on the front) so facilities can service it.\n\n## Scan to email\n- Scans are sent to your work email. Check Junk if it does not arrive within 5 minutes.\n",
 "data/kb/KB-007-software-install.md": "---\nid: KB-007\ntitle: Installing software\ncategory: software\nupdated: 2026-08-01\n---\n# Installing software\n\nStandard users do not have administrator rights. Approved software is installed through **Company Portal**.\n\n## Approved software (self-service)\n1. Open Company Portal from the Start menu.\n2. Search for the application (for example Zoom, Slack, Python 3, Tableau Reader, 7-Zip, Notepad++).\n3. Click **Install**. Installation runs in the background; a notification appears when it finishes.\n\n## Software not in Company Portal\n- Submit a software request through the IT Assistant with: application name, version, business justification, and whether a paid license is needed.\n- Requests are reviewed by IT Security within 3 business days. Paid licenses also need manager approval.\n\n## Install failed\n- Restart the laptop and retry from Company Portal.\n- Make sure at least 10 GB of disk space is free.\n- If the error repeats, note the error code shown in Company Portal and contact the Service Desk.\n\nUnapproved software (cracked tools, personal VPNs, browser extensions that read page data) is not allowed and is automatically removed.\n",
 "data/kb/KB-008-wifi.md": "---\nid: KB-008\ntitle: Office Wi-Fi connectivity\ncategory: network\nupdated: 2026-03-17\n---\n# Office Wi-Fi connectivity\n\n## Networks\n- `Bayview-Corp`: company laptops, uses your network username and password (certificate-based on managed devices).\n- `Bayview-Guest`: visitors and personal phones; requires accepting terms each day; no access to internal systems.\n\n## Cannot connect to Bayview-Corp\n1. Turn Wi-Fi off and on, or toggle Airplane mode.\n2. \"Forget\" the `Bayview-Corp` network and reconnect. If prompted, use your current password.\n3. If you recently changed your password, the old saved password will fail and can lock your account (KB-002).\n4. Restart the laptop.\n\n## Slow Wi-Fi\n- Move closer to an access point; conference rooms and the cafeteria have dedicated access points.\n- Close large uploads or cloud syncs (OneDrive) during meetings.\n- Run a speed test at https://speed.bayview.example and include the result when contacting IT.\n\n## Whole floor offline\nIf several people on the same floor cannot connect, this is likely an outage. Ask the IT Assistant to check service status; it will raise a ticket if none exists.\n",
 "data/kb/KB-009-slow-laptop.md": "---\nid: KB-009\ntitle: Laptop is slow or freezing\ncategory: hardware\nupdated: 2026-05-05\n---\n# Laptop is slow or freezing\n\n## Quick fixes\n1. Restart the laptop. Many users go weeks without restarting; updates and memory leaks build up.\n2. Open Task Manager (Ctrl+Shift+Esc) and sort by CPU and Memory. Close applications using unusually high resources.\n3. Check free disk space: keep at least 15% free. Empty Downloads and the Recycle Bin; set OneDrive files to \"Free up space\".\n4. Install pending updates: Settings > Windows Update, and Company Portal > Updates.\n\n## Browser\n- Too many tabs and extensions are a common cause. Remove extensions you do not use.\n\n## Hardware problems\n- Loud fan, overheating, battery swelling, or blue screens (BSOD) more than once a week: stop using the device if the battery is swollen and contact the Service Desk for a hardware ticket. Laptops older than 4 years are eligible for replacement.\n\n## Before contacting IT\nNote what you were doing when the slowdown started and whether it happens on battery and plugged in.\n",
 "data/kb/KB-010-access-requests.md": "---\nid: KB-010\ntitle: Requesting access to systems, shared drives and mailboxes\ncategory: access_request\nupdated: 2026-07-22\n---\n# Requesting access to systems, shared drives and mailboxes\n\nAccess follows the principle of least privilege: you get what your role needs, approved by the data owner.\n\n## How to request\n1. Tell the IT Assistant which system, folder or mailbox you need and why.\n2. A ticket is created and routed to the owner for approval (for example, the Finance share is owned by the Finance Director).\n3. After approval, access is granted within 1 business day. Sign out and back in for group changes to apply.\n\n## New hires\n- Managers submit onboarding requests at least 5 business days before the start date. Standard access (email, Wi-Fi, Company Portal, HR portal) is created automatically.\n\n## Elevated or sensitive access\n- Administrator rights, production databases, payroll and customer data require IT Security approval in addition to the data owner, and are reviewed every quarter.\n\n## Removing access\n- Managers must request removal when an employee changes role. Departing employees lose access automatically at 5 PM on their last day.\n",
 "data/kb/KB-011-security-incidents.md": "---\nid: KB-011\ntitle: Reporting phishing and security incidents\ncategory: security\nupdated: 2026-09-10\n---\n# Reporting phishing and security incidents\n\nSecurity incidents are always handled by a human analyst on the Security Operations team. Report quickly; you will not get in trouble for reporting.\n\n## Suspicious email (phishing)\n- Do not click links, open attachments, or reply.\n- Use the **Report Phish** button in Outlook. This sends the message to Security and removes it from your inbox.\n\n## If you clicked a link or entered your password\n1. Reset your password immediately (KB-001) from a different, trusted device if possible.\n2. Report it to the Service Desk as a security incident and say what you clicked and when.\n3. Watch for unexpected MFA prompts and deny them.\n\n## Lost or stolen device\n- Report within 1 hour so the device can be remotely locked and wiped. Include the time and place it was lost.\n\n## Other incidents\n- Ransomware messages, files suddenly renamed or encrypted, or antivirus alerts: disconnect from the network (turn off Wi-Fi, unplug the cable) and call the Service Desk immediately. Do not power off the device.\n",
 "data/kb/KB-012-sla-escalation.md": "---\nid: KB-012\ntitle: Service Desk priorities, SLAs and escalation\ncategory: policy\nupdated: 2026-06-30\n---\n# Service Desk priorities, SLAs and escalation\n\n| Priority | Definition | First response | Target resolution |\n|---|---|---|---|\n| P1 Critical | Security incident, outage affecting many users, or business-critical system down | 15 minutes | 4 hours |\n| P2 High | One user fully blocked from working, or VIP user affected | 1 hour | 8 business hours |\n| P3 Medium | Degraded service with a workaround available | 4 business hours | 3 business days |\n| P4 Low | Requests, questions, and how-to | 1 business day | 5 business days |\n\n## Queues\n- **Service Desk L1**: accounts, passwords, how-to, software requests.\n- **Desktop Support L2**: hardware, printers, laptop rebuilds.\n- **Network Ops**: Wi-Fi, VPN gateways, outages.\n- **Security Ops**: phishing, compromised accounts, lost devices. Always human-handled.\n- **Identity & Access**: access requests and MFA resets requiring identity verification.\n\n## Escalation rules\n- The IT Assistant escalates to a human when it cannot answer from the knowledge base, when an automated fix fails, when the user asks for a person, or for any security-related issue.\n- Escalated tickets include a summary, steps already tried, and the conversation transcript so the user does not have to repeat themselves.\n",
 "data/directory.json": "{\n  \"users\": [\n    {\"username\": \"jdoe\", \"name\": \"Jordan Doe\", \"email\": \"jdoe@bayview.example\", \"department\": \"Finance\", \"title\": \"Financial Analyst\", \"manager\": \"mrivera\", \"mfa_enrolled\": true, \"locked\": false, \"vip\": false},\n    {\"username\": \"asmith\", \"name\": \"Alex Smith\", \"email\": \"asmith@bayview.example\", \"department\": \"Sales\", \"title\": \"Account Executive\", \"manager\": \"mrivera\", \"mfa_enrolled\": true, \"locked\": true, \"vip\": false},\n    {\"username\": \"pchen\", \"name\": \"Priya Chen\", \"email\": \"pchen@bayview.example\", \"department\": \"Engineering\", \"title\": \"Data Engineer\", \"manager\": \"tkim\", \"mfa_enrolled\": true, \"locked\": false, \"vip\": false},\n    {\"username\": \"mrivera\", \"name\": \"Morgan Rivera\", \"email\": \"mrivera@bayview.example\", \"department\": \"Finance\", \"title\": \"Finance Director\", \"manager\": \"ceo\", \"mfa_enrolled\": true, \"locked\": false, \"vip\": true},\n    {\"username\": \"tkim\", \"name\": \"Taylor Kim\", \"email\": \"tkim@bayview.example\", \"department\": \"Engineering\", \"title\": \"Engineering Manager\", \"manager\": \"ceo\", \"mfa_enrolled\": true, \"locked\": false, \"vip\": false},\n    {\"username\": \"rpatel\", \"name\": \"Riley Patel\", \"email\": \"rpatel@bayview.example\", \"department\": \"Operations\", \"title\": \"Logistics Coordinator\", \"manager\": \"tkim\", \"mfa_enrolled\": false, \"locked\": false, \"vip\": false},\n    {\"username\": \"ceo\", \"name\": \"Casey Morgan\", \"email\": \"ceo@bayview.example\", \"department\": \"Executive\", \"title\": \"Chief Executive Officer\", \"manager\": null, \"mfa_enrolled\": true, \"locked\": false, \"vip\": true}\n  ]\n}\n",
 "data/services.json": "{\n  \"services\": [\n    {\"name\": \"vpn\", \"display\": \"GlobalConnect VPN\", \"status\": \"degraded\", \"note\": \"Gateway vpn-gw-02 is rejecting a portion of connections since 08:40. Network Ops is investigating; reconnecting usually lands on vpn-gw-01.\", \"queue\": \"Network Ops\"},\n    {\"name\": \"email\", \"display\": \"Email (Exchange Online)\", \"status\": \"operational\", \"note\": \"\", \"queue\": \"Service Desk L1\"},\n    {\"name\": \"sso\", \"display\": \"Single Sign-On portal\", \"status\": \"operational\", \"note\": \"\", \"queue\": \"Identity & Access\"},\n    {\"name\": \"wifi\", \"display\": \"Office Wi-Fi (Bayview-Corp)\", \"status\": \"operational\", \"note\": \"\", \"queue\": \"Network Ops\"},\n    {\"name\": \"printing\", \"display\": \"Secure print (BAYVIEW-SECURE)\", \"status\": \"operational\", \"note\": \"\", \"queue\": \"Desktop Support L2\"},\n    {\"name\": \"hr_portal\", \"display\": \"HR portal\", \"status\": \"outage\", \"note\": \"Scheduled maintenance 18:00-22:00 overran; vendor engaged. ETA 1 hour.\", \"queue\": \"Service Desk L1\"},\n    {\"name\": \"company_portal\", \"display\": \"Company Portal (software)\", \"status\": \"operational\", \"note\": \"\", \"queue\": \"Desktop Support L2\"}\n  ]\n}\n",
 "data/logs/auth.log": "2026-10-23T07:58:10Z INFO  sso login_success user=pchen app=outlook src=73.12.44.10\n2026-10-23T08:02:41Z ERROR sso login_failed user=asmith reason=bad_password app=outlook-mobile src=172.58.9.41\n2026-10-23T08:02:44Z ERROR sso login_failed user=asmith reason=bad_password app=outlook-mobile src=172.58.9.41\n2026-10-23T08:03:12Z ERROR sso login_failed user=asmith reason=bad_password app=outlook-mobile src=172.58.9.41\n2026-10-23T08:03:15Z ERROR sso login_failed user=asmith reason=bad_password app=outlook-mobile src=172.58.9.41\n2026-10-23T08:03:19Z ERROR sso login_failed user=asmith reason=bad_password app=outlook-mobile src=172.58.9.41\n2026-10-23T08:03:19Z WARN  sso account_locked user=asmith failed_attempts=5 window_min=15\n2026-10-23T08:10:07Z INFO  sso password_changed user=asmith method=self_service\n2026-10-23T08:20:55Z INFO  sso login_success user=tkim app=sso-portal src=98.4.201.7\n2026-10-23T08:33:30Z ERROR sso login_failed user=ceo reason=bad_password app=sso-portal src=185.220.101.34\n2026-10-23T08:33:41Z ERROR sso login_failed user=ceo reason=bad_password app=sso-portal src=185.220.101.34\n2026-10-23T08:33:52Z WARN  sso mfa_denied user=ceo app=sso-portal src=185.220.101.34\n2026-10-23T08:34:05Z WARN  sso impossible_travel user=ceo prev_src=50.77.130.9 new_src=185.220.101.34 minutes=6\n2026-10-23T08:44:02Z ERROR sso login_failed user=asmith reason=account_locked app=vpn src=172.58.9.41\n2026-10-23T09:01:16Z INFO  sso login_success user=rpatel app=hr-portal src=24.61.2.190\n",
 "data/logs/vpn.log": "2026-10-23T08:31:02Z INFO  vpn-gw-01 session_start user=pchen client=GlobalConnect/6.2.1 src=73.12.44.10\n2026-10-23T08:35:47Z INFO  vpn-gw-02 session_start user=tkim client=GlobalConnect/6.2.1 src=98.4.201.7\n2026-10-23T08:40:13Z ERROR vpn-gw-02 tls_handshake_failed user=jdoe reason=upstream_timeout src=66.9.13.88\n2026-10-23T08:40:58Z ERROR vpn-gw-02 tls_handshake_failed user=rpatel reason=upstream_timeout src=24.61.2.190\n2026-10-23T08:41:30Z WARN  vpn-gw-02 health_check latency_ms=4120 threshold_ms=1500\n2026-10-23T08:42:05Z ERROR vpn-gw-02 tls_handshake_failed user=jdoe reason=upstream_timeout src=66.9.13.88\n2026-10-23T08:42:40Z INFO  vpn-gw-01 session_start user=jdoe client=GlobalConnect/6.2.1 src=66.9.13.88\n2026-10-23T08:44:19Z ERROR vpn-gw-01 auth_failed user=asmith reason=invalid_credentials src=172.58.9.41\n2026-10-23T08:44:51Z ERROR vpn-gw-01 auth_failed user=asmith reason=invalid_credentials src=172.58.9.41\n2026-10-23T08:45:33Z ERROR vpn-gw-01 auth_failed user=asmith reason=account_locked src=172.58.9.41\n2026-10-23T08:47:12Z WARN  vpn-gw-02 health_check latency_ms=3890 threshold_ms=1500\n2026-10-23T08:49:02Z ERROR vpn-gw-02 tls_handshake_failed user=mrivera reason=upstream_timeout src=50.77.130.2\n2026-10-23T08:50:44Z ERROR vpn-gw-02 cert_validation_failed user=rpatel reason=device_cert_expired src=24.61.2.190\n2026-10-23T08:52:10Z INFO  vpn-gw-01 session_start user=mrivera client=GlobalConnect/6.2.1 src=50.77.130.2\n2026-10-23T08:55:31Z WARN  vpn-gw-02 health_check latency_ms=4410 threshold_ms=1500\n",
 "eval/scenarios.json": "{\n  \"description\": \"Realistic IT support scenarios with expected behaviour. expected_final is the agent that produces the final answer (knowledge | workflow | escalation). expected_category null = more than one category is acceptable.\",\n  \"scenarios\": [\n    {\"id\": \"K01\", \"requester\": \"jdoe\",   \"message\": \"How do I reset my password?\", \"expected_category\": \"password_reset\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-001\"},\n    {\"id\": \"K02\", \"requester\": \"pchen\",  \"message\": \"What are the password requirements? How long does it need to be?\", \"expected_category\": \"password_reset\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-001\"},\n    {\"id\": \"K03\", \"requester\": \"jdoe\",   \"message\": \"How do I set up Microsoft Authenticator on my new phone?\", \"expected_category\": \"account_access\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-003\"},\n    {\"id\": \"K04\", \"requester\": \"tkim\",   \"message\": \"VPN says certificate error when I try to connect, what do I do?\", \"expected_category\": \"vpn\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-004\"},\n    {\"id\": \"K05\", \"requester\": \"pchen\",  \"message\": \"How do I stop Outlook from asking for my password over and over?\", \"expected_category\": \"email\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-005\"},\n    {\"id\": \"K06\", \"requester\": \"jdoe\",   \"message\": \"My print job isn't coming out of the printer\", \"expected_category\": \"printer\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-006\"},\n    {\"id\": \"K07\", \"requester\": \"pchen\",  \"message\": \"How can I install Python on my laptop?\", \"expected_category\": \"software\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-007\"},\n    {\"id\": \"K08\", \"requester\": \"rpatel\", \"message\": \"I can't connect to the Bayview-Corp wifi on my laptop\", \"expected_category\": \"network\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-008\"},\n    {\"id\": \"K09\", \"requester\": \"tkim\",   \"message\": \"My laptop is really slow and keeps freezing\", \"expected_category\": \"hardware\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-009\"},\n    {\"id\": \"K10\", \"requester\": \"jdoe\",   \"message\": \"How do I get access to the Finance shared drive?\", \"expected_category\": \"access_request\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-010\"},\n    {\"id\": \"K11\", \"requester\": \"tkim\",   \"message\": \"Emails from one of our vendors aren't arriving in my inbox\", \"expected_category\": \"email\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-005\"},\n    {\"id\": \"K12\", \"requester\": \"jdoe\",   \"message\": \"How long does it take for an access request to be approved?\", \"expected_category\": \"access_request\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-010\"},\n    {\"id\": \"K13\", \"requester\": \"pchen\",  \"message\": \"A software install failed in Company Portal with an error code\", \"expected_category\": \"software\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-007\"},\n    {\"id\": \"K14\", \"requester\": \"rpatel\", \"message\": \"What is the GlobalConnect VPN portal address?\", \"expected_category\": \"vpn\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-004\"},\n    {\"id\": \"K15\", \"requester\": \"jdoe\",   \"message\": \"How do I scan a document and send it to my email?\", \"expected_category\": \"printer\", \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-006\"},\n    {\"id\": \"K16\", \"requester\": \"tkim\",   \"message\": \"What is the response time for a P2 ticket?\", \"expected_category\": null, \"expected_final\": \"knowledge\", \"expected_action\": \"none\", \"expected_doc\": \"KB-012\"},\n\n    {\"id\": \"W01\", \"requester\": \"asmith\", \"message\": \"I'm locked out of my account, please unlock it\", \"expected_category\": \"account_lockout\", \"expected_final\": \"workflow\", \"expected_action\": \"unlock_account\", \"expected_doc\": null},\n    {\"id\": \"W02\", \"requester\": \"asmith\", \"message\": \"Locked out again right after changing my password. Can you unlock me?\", \"expected_category\": \"account_lockout\", \"expected_final\": \"workflow\", \"expected_action\": \"unlock_account\", \"expected_doc\": null},\n    {\"id\": \"W03\", \"requester\": \"jdoe\",   \"message\": \"I forgot my password, please reset it\", \"expected_category\": \"password_reset\", \"expected_final\": \"workflow\", \"expected_action\": \"reset_password\", \"expected_doc\": null},\n    {\"id\": \"W04\", \"requester\": \"pchen\",  \"message\": \"My password expired, can you reset it for me?\", \"expected_category\": \"password_reset\", \"expected_final\": \"workflow\", \"expected_action\": \"reset_password\", \"expected_doc\": null},\n    {\"id\": \"W05\", \"requester\": \"pchen\",  \"message\": \"Is the VPN down right now?\", \"expected_category\": \"vpn\", \"expected_final\": \"workflow\", \"expected_action\": \"check_service_status\", \"expected_doc\": null},\n    {\"id\": \"W06\", \"requester\": \"tkim\",   \"message\": \"Is the HR portal down? It won't load\", \"expected_category\": null, \"expected_final\": \"workflow\", \"expected_action\": \"check_service_status\", \"expected_doc\": null},\n    {\"id\": \"W07\", \"requester\": \"jdoe\",   \"message\": \"Is anyone else having email problems or is it just me?\", \"expected_category\": \"email\", \"expected_final\": \"workflow\", \"expected_action\": \"check_service_status\", \"expected_doc\": null},\n    {\"id\": \"W08\", \"requester\": \"asmith\", \"message\": \"Why does my account keep locking? Can you check the logs?\", \"expected_category\": \"account_lockout\", \"expected_final\": \"workflow\", \"expected_action\": \"analyze_logs\", \"expected_doc\": null},\n    {\"id\": \"W09\", \"requester\": \"tkim\",   \"message\": \"Can you check the status of all IT services?\", \"expected_category\": null, \"expected_final\": \"workflow\", \"expected_action\": \"check_service_status\", \"expected_doc\": null},\n\n    {\"id\": \"E01\", \"requester\": \"jdoe\",   \"message\": \"Please reset the password for asmith\", \"expected_category\": \"password_reset\", \"expected_final\": \"escalation\", \"expected_action\": \"reset_password\", \"expected_doc\": null},\n    {\"id\": \"E02\", \"requester\": \"rpatel\", \"message\": \"I'm locked out, can you unlock my account?\", \"expected_category\": \"account_lockout\", \"expected_final\": \"escalation\", \"expected_action\": \"unlock_account\", \"expected_doc\": null},\n    {\"id\": \"E03\", \"requester\": \"ceo\",    \"message\": \"Please reset my password, I forgot it\", \"expected_category\": \"password_reset\", \"expected_final\": \"escalation\", \"expected_action\": \"reset_password\", \"expected_doc\": null},\n    {\"id\": \"E04\", \"requester\": \"rpatel\", \"message\": \"Forgot my password and need a reset\", \"expected_category\": \"password_reset\", \"expected_final\": \"escalation\", \"expected_action\": \"reset_password\", \"expected_doc\": null},\n\n    {\"id\": \"S01\", \"requester\": \"jdoe\",   \"message\": \"I clicked a link in a fake invoice email and typed my password\", \"expected_category\": \"security\", \"expected_final\": \"escalation\", \"expected_action\": \"none\", \"expected_doc\": null},\n    {\"id\": \"S02\", \"requester\": \"pchen\",  \"message\": \"I left my company laptop on the train this morning and it's gone\", \"expected_category\": \"security\", \"expected_final\": \"escalation\", \"expected_action\": \"none\", \"expected_doc\": null},\n    {\"id\": \"S03\", \"requester\": \"tkim\",   \"message\": \"I keep getting MFA prompts that I never started\", \"expected_category\": \"security\", \"expected_final\": \"escalation\", \"expected_action\": \"none\", \"expected_doc\": null},\n    {\"id\": \"S04\", \"requester\": \"jdoe\",   \"message\": \"All my files got renamed and there's a ransom note on my desktop\", \"expected_category\": \"security\", \"expected_final\": \"escalation\", \"expected_action\": \"none\", \"expected_doc\": null},\n    {\"id\": \"S05\", \"requester\": \"pchen\",  \"message\": \"I just want to talk to a real person please\", \"expected_category\": null, \"expected_final\": \"escalation\", \"expected_action\": \"none\", \"expected_doc\": null},\n    {\"id\": \"S06\", \"requester\": \"rpatel\", \"message\": \"How do I book a conference room for the team offsite?\", \"expected_category\": null, \"expected_final\": \"escalation\", \"expected_action\": \"none\", \"expected_doc\": null},\n    {\"id\": \"S07\", \"requester\": \"tkim\",   \"message\": \"What's on the cafeteria menu today?\", \"expected_category\": null, \"expected_final\": \"escalation\", \"expected_action\": \"none\", \"expected_doc\": null},\n    {\"id\": \"S08\", \"requester\": \"jdoe\",   \"message\": \"Our whole floor lost wifi and nobody can work, is it down?\", \"expected_category\": \"network\", \"expected_final\": \"escalation\", \"expected_action\": \"check_service_status\", \"expected_doc\": null}\n  ]\n}\n"
}''')
for path, text in DATA.items():
    p = pathlib.Path(path); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text, encoding='utf-8')
print(f'wrote {len(DATA)} data files')

In [ ]:
%%writefile helpdesk/__init__.py
"""Bayview IT Assistant: a multi-agent IT support system built with LangGraph + Gemini.

Agents: Intake -> (Knowledge | Workflow | Escalation). See helpdesk/graph.py.
"""

__version__ = "1.0.0"


In [ ]:
%%writefile helpdesk/config.py
"""Central configuration.

The Gemini API key is read from a secret named ``GEMINI_API_KEY``, checked in this order:
  1. Google Colab secrets (``google.colab.userdata``), when running in Colab
  2. The ``GEMINI_API_KEY`` environment variable
  3. A ``.env`` file in the project root

If no key is found the system runs in *offline mode*: deterministic keyword classification,
hashed bag-of-words embeddings and extractive answers. Offline mode exists so the graph,
tools and tests can be exercised without network access or API quota; it is clearly
labelled in every response and in evaluation reports.
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from pathlib import Path

SECRET_NAME = "GEMINI_API_KEY"

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
KB_DIR = DATA_DIR / "kb"
LOG_DIR = DATA_DIR / "logs"
RUNTIME_DIR = PROJECT_ROOT / "runtime"  # mutable state: ticket DB, directory copy, vector store


def _load_dotenv() -> None:
    try:
        from dotenv import load_dotenv
    except ImportError:
        return
    load_dotenv(PROJECT_ROOT / ".env", override=False)


def get_api_key() -> str | None:
    """Return the Gemini key from Colab secrets, env, or .env (never logged or printed)."""
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get(SECRET_NAME)
        if key:
            return key
    except Exception:
        # Not in Colab, or the secret exists but notebook access was not granted.
        pass
    _load_dotenv()
    return os.environ.get(SECRET_NAME) or None


@dataclass
class Settings:
    api_key: str | None = field(default_factory=get_api_key)
    # "gemini-flash-latest" tracks Google's current stable Flash model, so the project keeps
    # working when a pinned version is retired (gemini-2.5-* was shut down 2026-10-16).
    chat_model: str = field(default_factory=lambda: os.environ.get("GEMINI_MODEL", "gemini-flash-latest"))
    embedding_models: tuple[str, ...] = field(
        default_factory=lambda: tuple(
            m.strip()
            for m in os.environ.get("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2,gemini-embedding-001").split(",")
            if m.strip()
        )
    )
    force_offline: bool = field(default_factory=lambda: os.environ.get("HELPDESK_OFFLINE", "").lower() in ("1", "true", "yes"))
    # Retrieval: chunks below this cosine similarity are treated as "not in the knowledge base".
    retrieval_top_k: int = 4
    # Gemini and offline embeddings live on different similarity scales, so each mode has its own default.
    min_relevance_override: float | None = field(
        default_factory=lambda: float(os.environ["HELPDESK_MIN_RELEVANCE"]) if os.environ.get("HELPDESK_MIN_RELEVANCE") else None
    )
    # Seconds between Gemini calls; raise this on the free tier to stay under requests-per-minute limits.
    min_call_interval: float = field(default_factory=lambda: float(os.environ.get("GEMINI_MIN_INTERVAL", "0")))
    # Optional: escalations also open a GitHub issue when both are set.
    github_token: str | None = field(default_factory=lambda: os.environ.get("GITHUB_TOKEN") or None)
    github_repo: str | None = field(default_factory=lambda: os.environ.get("GITHUB_REPO") or None)

    @property
    def offline(self) -> bool:
        return self.force_offline or not self.api_key

    @property
    def min_relevance(self) -> float:
        if self.min_relevance_override is not None:
            return self.min_relevance_override
        # Offline value calibrated on eval/scenarios.json (relevant >= 0.16, off-topic <= 0.12).
        return 0.15 if self.offline else 0.55

    @property
    def mode_label(self) -> str:
        return "offline (no GEMINI_API_KEY)" if self.offline else f"gemini:{self.chat_model}"


_settings: Settings | None = None


def settings() -> Settings:
    global _settings
    if _settings is None:
        _settings = Settings()
    return _settings


def reset_settings() -> None:
    """Re-read configuration (used by tests and notebooks after changing env vars)."""
    global _settings
    _settings = None


In [ ]:
%%writefile helpdesk/privacy.py
"""PII redaction applied before text is sent to the LLM, stored in tickets, or logged.

Usernames and work emails are kept (agents need them to act); secrets and personal
identifiers are masked. This is a guardrail, not a DLP product: the scaling document
describes replacing it with a managed service such as Google Cloud DLP.
"""

from __future__ import annotations

import re

_PATTERNS: list[tuple[str, re.Pattern[str]]] = [
    # "my password is hunter2" / "pwd: hunter2" -> never forward a password anywhere.
    ("PASSWORD", re.compile(r"(?i)\b(pass(?:word|code|wd)?|pwd)\b(\s*(?:is|was|=|:)\s*)(\S+)")),
    ("SSN", re.compile(r"\b\d{3}-\d{2}-\d{4}\b")),
    ("CARD", re.compile(r"\b(?:\d[ -]?){13,16}\b")),
    ("PHONE", re.compile(r"(?<!\d)(?:\+?1[ .-]?)?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}(?!\d)")),
    ("EMAIL", re.compile(r"\b[\w.+-]+@(?!bayview\.example)[\w-]+\.[\w.]+\b")),  # personal emails only
]


def redact(text: str) -> tuple[str, list[str]]:
    """Return (redacted_text, list_of_redaction_types_applied)."""
    found: list[str] = []
    for label, pattern in _PATTERNS:
        if label == "PASSWORD":
            text, n = pattern.subn(lambda m: f"{m.group(1)}{m.group(2)}[REDACTED-PASSWORD]", text)
        else:
            text, n = pattern.subn(f"[REDACTED-{label}]", text)
        if n:
            found.append(label)
    return text, found


In [ ]:
%%writefile helpdesk/schemas.py
"""Typed contracts between agents. Gemini is asked to return JSON matching these models."""

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field

Category = Literal[
    "password_reset", "account_lockout", "account_access", "vpn", "email", "printer",
    "software", "network", "hardware", "access_request", "security", "other",
]
Action = Literal["none", "reset_password", "unlock_account", "check_service_status", "analyze_logs"]
Service = Literal["none", "vpn", "email", "sso", "wifi", "printing", "hr_portal", "company_portal"]


class IntakeResult(BaseModel):
    category: Category = Field(description="Best-fitting IT support category.")
    request_type: Literal["question", "action", "incident"] = Field(
        description="question = user wants information/how-to; action = user wants something done to their account or a system checked; incident = security event.")
    action: Action = Field(description="Automation that would resolve the request, or 'none'.")
    target_username: str = Field(description="Username the action is for, if the user named someone else. Empty string when it is the requester.")
    service: Service = Field(description="System the request is about, or 'none'.")
    urgency: Literal["low", "medium", "high", "critical"]
    wants_human: bool = Field(description="True if the user explicitly asks for a person.")
    summary: str = Field(description="One-sentence neutral summary of the request for a technician.")
    # No defaults or numeric bounds: the Gemini API rejects them in response schemas.
    confidence: float = Field(description="Confidence in the category, 0-1.")


class KnowledgeAnswer(BaseModel):
    answer: str = Field(description="Answer for the user, in short steps, using only the provided context.")
    cited_ids: list[str] = Field(description="Chunk ids (e.g. KB-004#2) the answer relies on.")
    grounded: bool = Field(description="False if the context does not contain the answer.")


class HandoffSummary(BaseModel):
    title: str = Field(description="Ticket title, under 80 characters.")
    summary: str = Field(description="2-3 sentence summary for the technician: problem, impact, relevant facts.")
    steps_tried: list[str] = Field(description="What the user or assistant already tried.")


In [ ]:
%%writefile helpdesk/llm.py
"""Thin wrapper around the Gemini API (google-genai SDK) with retries, pacing and metrics.

Agents call ``structured()`` for typed JSON output (validated by Pydantic), ``text()`` for
free-form answers and ``embed()`` for vectors. When the client is offline, ``embed()`` falls
back to a deterministic hashed bag-of-words embedding and agents use their own rule-based
paths (see ``helpdesk/offline.py``).
"""

from __future__ import annotations

import hashlib
import math
import re
import threading
import time
from dataclasses import dataclass, field
from typing import TypeVar

import numpy as np
from pydantic import BaseModel

from .config import Settings, settings

T = TypeVar("T", bound=BaseModel)

OFFLINE_DIM = 1024
EMBED_DIM = 768
_RETRYABLE = {429, 500, 502, 503, 504}


@dataclass
class CallStats:
    llm_calls: int = 0
    embed_calls: int = 0
    llm_seconds: float = 0.0
    retries: int = 0
    errors: list[str] = field(default_factory=list)


class LLM:
    def __init__(self, cfg: Settings | None = None):
        self.cfg = cfg or settings()
        self.stats = CallStats()
        self._lock = threading.Lock()
        self._last_call = 0.0
        self._client = None
        self._embedding_model: str | None = None
        if not self.cfg.offline:
            from google import genai

            self._client = genai.Client(api_key=self.cfg.api_key)

    @property
    def offline(self) -> bool:
        return self._client is None

    # ------------------------------------------------------------------ generation
    def structured(self, prompt: str, schema: type[T], system: str) -> T:
        from google.genai import types

        config = types.GenerateContentConfig(
            system_instruction=system,
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=schema,
        )
        resp = self._generate(prompt, config)
        parsed = getattr(resp, "parsed", None)
        if isinstance(parsed, schema):
            return parsed
        return schema.model_validate_json(resp.text)

    def text(self, prompt: str, system: str, temperature: float = 0.2) -> str:
        from google.genai import types

        config = types.GenerateContentConfig(system_instruction=system, temperature=temperature)
        return (self._generate(prompt, config).text or "").strip()

    def _generate(self, prompt: str, config):
        if self.offline:
            raise RuntimeError("LLM is offline; agents must use their offline path.")
        start = time.perf_counter()
        try:
            return self._with_retries(
                lambda: self._client.models.generate_content(model=self.cfg.chat_model, contents=prompt, config=config)
            )
        finally:
            self.stats.llm_calls += 1
            self.stats.llm_seconds += time.perf_counter() - start

    # ------------------------------------------------------------------ embeddings
    def embed(self, texts: list[str], task: str = "document") -> np.ndarray:
        """Return an L2-normalised (n, d) matrix. task is 'document' or 'query'."""
        if self.offline:
            return np.vstack([offline_embed(t) for t in texts])
        vectors: list[list[float]] = []
        for i in range(0, len(texts), 100):  # API batch limit
            vectors.extend(self._embed_batch(texts[i : i + 100], task))
        self.stats.embed_calls += 1
        arr = np.asarray(vectors, dtype=np.float32)
        return arr / np.linalg.norm(arr, axis=1, keepdims=True).clip(min=1e-9)

    def _embed_batch(self, batch: list[str], task: str) -> list[list[float]]:
        from google.genai import errors, types

        task_type = "RETRIEVAL_QUERY" if task == "query" else "RETRIEVAL_DOCUMENT"
        candidates = [self._embedding_model] if self._embedding_model else list(self.cfg.embedding_models)
        last_err: Exception | None = None
        for model in candidates:
            # Not every embedding model accepts task_type; retry once without it.
            for config in (
                types.EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM),
                types.EmbedContentConfig(output_dimensionality=EMBED_DIM),
            ):
                try:
                    resp = self._with_retries(
                        lambda: self._client.models.embed_content(model=model, contents=batch, config=config)
                    )
                    self._embedding_model = model
                    return [e.values for e in resp.embeddings]
                except errors.APIError as e:
                    last_err = e
                    if e.code == 404:
                        break  # model unavailable: try the next candidate
                    if e.code != 400:
                        raise
        raise RuntimeError(f"No embedding model worked ({candidates}): {last_err}")

    @property
    def embedding_model(self) -> str:
        return "offline-hashed-bow" if self.offline else (self._embedding_model or self.cfg.embedding_models[0])

    # ------------------------------------------------------------------ plumbing
    def _with_retries(self, fn, attempts: int = 5):
        from google.genai import errors

        delay = 2.0
        for attempt in range(attempts):
            self._pace()
            try:
                return fn()
            except errors.APIError as e:
                if e.code not in _RETRYABLE or attempt == attempts - 1:
                    self.stats.errors.append(f"{e.code}: {str(e)[:120]}")
                    raise
                self.stats.retries += 1
                time.sleep(delay)
                delay = min(delay * 2, 30)

    def _pace(self) -> None:
        if self.cfg.min_call_interval <= 0:
            return
        with self._lock:
            wait = self._last_call + self.cfg.min_call_interval - time.monotonic()
            if wait > 0:
                time.sleep(wait)
            self._last_call = time.monotonic()


# ---------------------------------------------------------------------- offline embedding
_SYNONYMS = {
    "pw": "password", "passwd": "password", "pwd": "password", "passcode": "password",
    "locked": "lockout", "lock": "lockout", "unlock": "lockout",
    "globalconnect": "vpn", "tunnel": "vpn",
    "outlook": "email", "mail": "email", "inbox": "email", "mailbox": "email",
    "wifi": "wi-fi", "wireless": "wi-fi", "internet": "wi-fi",
    "print": "printer", "printing": "printer", "scan": "printer", "scanner": "printer", "toner": "printer",
    "install": "software", "app": "software", "application": "software", "program": "software",
    "slow": "performance", "freezing": "performance", "freeze": "performance", "laggy": "performance",
    "authenticator": "mfa", "2fa": "mfa", "two-factor": "mfa",
    "phish": "phishing", "phishy": "phishing", "suspicious": "phishing", "scam": "phishing",
    "permission": "access", "permissions": "access", "share": "access", "drive": "access",
}
_STOP = set("a an the to of and or in on for my i me is it its be am are was can't cannot can how do does what why with this that at from your you we our".split())


def _tokens(text: str) -> list[str]:
    words = re.findall(r"[a-z0-9][a-z0-9\-]*", text.lower())
    out = []
    for w in words:
        if w in _STOP:
            continue
        w = _SYNONYMS.get(w, w)
        for suffix in ("ing", "ed", "es", "s"):
            if len(w) > 4 and w.endswith(suffix):
                w = w[: -len(suffix)]
                break
        out.append(_SYNONYMS.get(w, w))
    return out


def offline_embed(text: str) -> np.ndarray:
    toks = _tokens(text)
    feats = toks + [f"{a}_{b}" for a, b in zip(toks, toks[1:])]
    vec = np.zeros(OFFLINE_DIM, dtype=np.float32)
    for f in feats:
        vec[int(hashlib.md5(f.encode()).hexdigest(), 16) % OFFLINE_DIM] += 1.0
    vec = np.log1p(vec)
    norm = math.sqrt(float(vec @ vec)) or 1.0
    return vec / norm


_llm: LLM | None = None


def get_llm() -> LLM:
    global _llm
    if _llm is None:
        _llm = LLM()
    return _llm


def reset_llm() -> None:
    global _llm
    _llm = None


In [ ]:
%%writefile helpdesk/offline.py
"""Rule-based stand-ins for the LLM calls, used only when no GEMINI_API_KEY is available.

These make the system runnable end to end (CI, demos without quota). They are also a useful
baseline: the evaluation report compares Gemini against these rules.
"""

from __future__ import annotations

import re

from . import store
from .schemas import IntakeResult

_CATEGORY_RULES: list[tuple[str, str]] = [
    ("security", r"phish|suspicious (e-?mail|link)|clicked (a|the) link|hacked|compromis|stolen|ransom|virus|malware"
                 r"|(lost|left) my (company )?(laptop|phone)|(didn'?t|never|did not) (start|request|initiate)"),
    ("account_lockout", r"locked|lockout|lock(ed)? out|unlock"),
    ("password_reset", r"password|forgot|reset my|expired"),
    ("account_access", r"\bmfa\b|authenticator|2fa|two.factor|new phone"),
    ("vpn", r"\bvpn\b|globalconnect|gateway|remote access"),
    ("email", r"outlook|e-?mail|inbox|mailbox|calendar"),
    ("printer", r"print|scan|toner|paper jam"),
    ("software", r"install|software|\bapp\b|application|license|company portal|zoom|slack|tableau|python"),
    ("network", r"wi-?fi|wireless|internet|network|ethernet"),
    ("hardware", r"slow|freez|laptop|battery|blue screen|bsod|overheat|fan|monitor|keyboard"),
    ("access_request", r"access to|permission|shared (drive|folder|mailbox)|need access"),
]


def classify(message: str, requester: str) -> IntakeResult:
    text = message.lower()
    category = next((c for c, pat in _CATEGORY_RULES if re.search(pat, text)), "other")
    wants_human = bool(re.search(r"(talk|speak) to (a |an )?(human|person|agent|someone|technician)|real person", text))

    action, request_type = "none", "question"
    asks_to_do = re.search(r"\b(please|can you|could you|i need you to|reset|unlock|check|is the|are the|is there)\b", text)
    how_to = re.search(r"\bhow (do|can|to)\b|\bwhat (is|are)\b|\bwhere\b", text)
    if re.search(r"\blogs?\b|why (do|does|did|am|is) .*(lock|fail|keep)", text) and asks_to_do:
        action, request_type = "analyze_logs", "action"
    elif category == "account_lockout" and not how_to:
        action, request_type = "unlock_account", "action"
    elif category == "password_reset" and re.search(r"reset|forgot|expired", text) and not how_to:
        action, request_type = "reset_password", "action"
    elif re.search(r"\b(down|outage|status|anyone else|everyone|whole (floor|team|office))\b", text):
        action, request_type = "check_service_status", "action"
    if category == "security":
        action, request_type = "none", "incident"

    service = "none"
    for svc, pat in [("vpn", r"vpn|globalconnect"), ("email", r"outlook|e-?mail"), ("wifi", r"wi-?fi|wireless"),
                     ("printing", r"print"), ("hr_portal", r"hr portal|payroll portal|benefits"), ("sso", r"\bsso\b|single sign"),
                     ("company_portal", r"company portal")]:
        if re.search(pat, text):
            service = svc
            break

    # Only treat a word as a target username if it exists in the directory ("unlock jdoe", "for asmith").
    target = ""
    for word in re.findall(r"\b(?:for|unlock|reset|user(?:name)?)\s+([a-z]{2,10})\b", text):
        if store.get_user(word) and word != requester:
            target = word
            break

    urgency = "medium"
    if category == "security" or re.search(r"urgent|asap|emergency|(can'?t|cannot|nobody can|no one can) work|critical|client meeting", text):
        urgency = "high" if category != "security" else "critical"
    elif request_type == "question":
        urgency = "low"

    return IntakeResult(category=category, request_type=request_type, action=action, target_username=target,
                        service=service, urgency=urgency, wants_human=wants_human,
                        summary=message.strip()[:160], confidence=0.6 if category != "other" else 0.3)


def extractive_answer(question: str, hits) -> str:
    """Return the most relevant KB section, lightly trimmed, as the answer."""
    best = hits[0].chunk
    body = best.text.split("\n", 1)[1].strip() if "\n" in best.text else best.text
    return f"Here's what our knowledge base says ({best.title} › {best.section}):\n\n{body}"


In [ ]:
%%writefile helpdesk/knowledge_base.py
"""RAG knowledge base: Markdown KB articles -> section chunks -> embeddings -> Chroma.

Chunking is by ``##`` section (with the article title prepended) so each chunk is a
self-contained answer unit and citations can point at a specific section.
The collection is rebuilt when the KB content or the embedding model changes
(tracked with a content hash stored in the collection metadata).
"""

from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass
from pathlib import Path

import chromadb

from .config import KB_DIR, RUNTIME_DIR
from .llm import LLM, get_llm


@dataclass
class Chunk:
    id: str
    doc_id: str
    title: str
    section: str
    category: str
    text: str


@dataclass
class Hit:
    chunk: Chunk
    score: float  # cosine similarity, 1.0 = identical

    @property
    def citation(self) -> str:
        return f"{self.chunk.doc_id} · {self.chunk.title} › {self.chunk.section}"


def _parse_front_matter(raw: str) -> tuple[dict[str, str], str]:
    meta: dict[str, str] = {}
    if raw.startswith("---"):
        _, fm, body = raw.split("---", 2)
        for line in fm.strip().splitlines():
            k, _, v = line.partition(":")
            meta[k.strip()] = v.strip()
        return meta, body
    return meta, raw


def load_chunks(kb_dir: Path = KB_DIR) -> list[Chunk]:
    chunks: list[Chunk] = []
    for path in sorted(kb_dir.glob("*.md")):
        meta, body = _parse_front_matter(path.read_text(encoding="utf-8"))
        doc_id, title, category = meta["id"], meta["title"], meta.get("category", "general")
        # Split on level-2 headings; the text before the first one is the "Overview".
        parts = re.split(r"^## +(.+)$", body, flags=re.M)
        intro = re.sub(r"^# .+$", "", parts[0], flags=re.M).strip()
        sections = [("Overview", intro)] if intro else []
        sections += [(parts[i].strip(), parts[i + 1].strip()) for i in range(1, len(parts) - 1, 2)]
        for n, (section, text) in enumerate(sections):
            chunks.append(Chunk(f"{doc_id}#{n}", doc_id, title, section, category, f"{title} — {section}\n{text}"))
    return chunks


class KnowledgeBase:
    def __init__(self, llm: LLM | None = None, persist: bool = True):
        self.llm = llm or get_llm()
        self.chunks = load_chunks()
        self._by_id = {c.id: c for c in self.chunks}
        if persist:
            RUNTIME_DIR.mkdir(exist_ok=True)
            client = chromadb.PersistentClient(path=str(RUNTIME_DIR / "chroma"))
        else:
            client = chromadb.EphemeralClient()
        # One collection per embedding model so switching models never mixes vector spaces.
        name = "kb_" + re.sub(r"[^a-zA-Z0-9_-]", "_", self.llm.embedding_model)[:50]
        self.collection = client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})
        self._ensure_indexed()

    def _content_hash(self) -> str:
        return hashlib.sha256("".join(c.id + c.text for c in self.chunks).encode()).hexdigest()[:16]

    def _ensure_indexed(self) -> None:
        digest = self._content_hash()
        existing = self.collection.get(ids=["__meta__"], include=["metadatas"])
        if existing["ids"] and existing["metadatas"][0].get("hash") == digest:
            return
        if self.collection.count():
            self.collection.delete(ids=self.collection.get()["ids"])
        vectors = self.llm.embed([c.text for c in self.chunks], task="document")
        self.collection.add(
            ids=[c.id for c in self.chunks],
            embeddings=vectors.tolist(),
            documents=[c.text for c in self.chunks],
            metadatas=[{"doc_id": c.doc_id, "title": c.title, "section": c.section, "category": c.category} for c in self.chunks],
        )
        self.collection.add(ids=["__meta__"], embeddings=[vectors[0].tolist()], documents=[""], metadatas=[{"hash": digest}])

    def search(self, query: str, k: int = 4) -> list[Hit]:
        qvec = self.llm.embed([query], task="query")[0]
        res = self.collection.query(query_embeddings=[qvec.tolist()], n_results=k + 1, include=["distances"])
        hits = []
        for cid, dist in zip(res["ids"][0], res["distances"][0]):
            if cid == "__meta__":
                continue
            hits.append(Hit(self._by_id[cid], round(1.0 - dist, 4)))
        return hits[:k]

    @property
    def size(self) -> dict[str, int]:
        return {"articles": len({c.doc_id for c in self.chunks}), "chunks": len(self.chunks)}


_kb: KnowledgeBase | None = None


def get_kb() -> KnowledgeBase:
    global _kb
    if _kb is None:
        _kb = KnowledgeBase()
    return _kb


def reset_kb() -> None:
    global _kb
    _kb = None


In [ ]:
%%writefile helpdesk/store.py
"""Mutable mock back-office systems: user directory (identity provider) and ticket database.

The seed data in ``data/`` is copied into ``runtime/`` on first use so demos and
evaluations can change state (unlock accounts, open tickets) without editing the seeds.
``reset_runtime()`` restores a clean state.
"""

from __future__ import annotations

import json
import shutil
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from .config import DATA_DIR, RUNTIME_DIR

DIRECTORY_PATH = RUNTIME_DIR / "directory.json"
TICKETS_DB = RUNTIME_DIR / "tickets.db"


def _now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def reset_runtime() -> None:
    """Restore seed data and clear tickets (keeps the vector store, which is content-hashed)."""
    RUNTIME_DIR.mkdir(exist_ok=True)
    shutil.copy(DATA_DIR / "directory.json", DIRECTORY_PATH)
    # Clear rows instead of deleting the file: Windows refuses to delete an SQLite file another connection holds.
    with _db() as con:
        con.execute("DELETE FROM tickets")
        con.execute("DELETE FROM sqlite_sequence WHERE name = 'tickets'")


# ---------------------------------------------------------------- directory (mock identity provider)
def _load_directory() -> dict:
    if not DIRECTORY_PATH.exists():
        reset_runtime()
    return json.loads(DIRECTORY_PATH.read_text(encoding="utf-8"))


def get_user(username: str) -> dict | None:
    username = (username or "").lower().strip()
    return next((u for u in _load_directory()["users"] if u["username"] == username), None)


def update_user(username: str, **changes) -> dict:
    data = _load_directory()
    for u in data["users"]:
        if u["username"] == username:
            u.update(changes)
            DIRECTORY_PATH.write_text(json.dumps(data, indent=2), encoding="utf-8")
            return u
    raise KeyError(username)


def list_users() -> list[dict]:
    return _load_directory()["users"]


# ---------------------------------------------------------------- tickets
_SCHEMA = """
CREATE TABLE IF NOT EXISTS tickets (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    created_at  TEXT NOT NULL,
    requester   TEXT NOT NULL,
    category    TEXT NOT NULL,
    priority    TEXT NOT NULL,
    queue       TEXT NOT NULL,
    status      TEXT NOT NULL,
    title       TEXT NOT NULL,
    summary     TEXT NOT NULL,
    steps_tried TEXT NOT NULL,
    transcript  TEXT NOT NULL,
    resolved_by TEXT,
    external_url TEXT
)
"""


def _db() -> sqlite3.Connection:
    RUNTIME_DIR.mkdir(exist_ok=True)
    con = sqlite3.connect(TICKETS_DB)
    con.row_factory = sqlite3.Row
    con.execute(_SCHEMA)
    return con


def create_ticket(*, requester: str, category: str, priority: str, queue: str, title: str, summary: str,
                  steps_tried: list[str] | None = None, transcript: str = "", status: str = "open",
                  resolved_by: str | None = None) -> dict:
    with _db() as con:
        cur = con.execute(
            "INSERT INTO tickets (created_at, requester, category, priority, queue, status, title, summary,"
            " steps_tried, transcript, resolved_by) VALUES (?,?,?,?,?,?,?,?,?,?,?)",
            (_now(), requester, category, priority, queue, status, title, summary,
             json.dumps(steps_tried or []), transcript, resolved_by),
        )
        return get_ticket(cur.lastrowid, con)


def get_ticket(ticket_id: int, con: sqlite3.Connection | None = None) -> dict | None:
    con = con or _db()
    row = con.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone()
    if not row:
        return None
    t = dict(row)
    t["steps_tried"] = json.loads(t["steps_tried"])
    t["number"] = f"INC{t['id']:06d}"
    return t


def set_external_url(ticket_id: int, url: str) -> None:
    with _db() as con:
        con.execute("UPDATE tickets SET external_url = ? WHERE id = ?", (url, ticket_id))


def list_tickets(status: str | None = None, queue: str | None = None, limit: int = 50) -> list[dict]:
    sql, args = "SELECT id FROM tickets WHERE 1=1", []
    if status:
        sql += " AND status = ?"
        args.append(status)
    if queue:
        sql += " AND queue = ?"
        args.append(queue)
    sql += " ORDER BY id DESC LIMIT ?"
    args.append(limit)
    con = _db()
    return [get_ticket(r["id"], con) for r in con.execute(sql, args).fetchall()]


def update_ticket_status(ticket_id: int, status: str, resolved_by: str | None = None) -> dict | None:
    with _db() as con:
        con.execute("UPDATE tickets SET status = ?, resolved_by = COALESCE(?, resolved_by) WHERE id = ?",
                    (status, resolved_by, ticket_id))
        return get_ticket(ticket_id, con)


def load_services() -> list[dict]:
    return json.loads((DATA_DIR / "services.json").read_text(encoding="utf-8"))["services"]


def read_log(name: str) -> str:
    path: Path = DATA_DIR / "logs" / f"{name}.log"
    if not path.resolve().is_relative_to((DATA_DIR / "logs").resolve()) or not path.exists():
        raise FileNotFoundError(name)
    return path.read_text(encoding="utf-8")


In [ ]:
%%writefile helpdesk/tools.py
"""Workflow tools: the automations the Workflow Agent (and the MCP server) can run.

Every tool returns a ``ToolResult`` so callers can tell success from a refusal that must
be escalated. Tools enforce their own guardrails (self-service only, MFA required,
security holds) instead of trusting the LLM to have checked.
"""

from __future__ import annotations

import json
import re
import urllib.request
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field

from . import store
from .config import settings


@dataclass
class ToolResult:
    tool: str
    ok: bool
    message: str
    data: dict = field(default_factory=dict)
    needs_escalation: bool = False
    escalation_reason: str = ""

    def to_dict(self) -> dict:
        return asdict(self)


# ---------------------------------------------------------------- log analysis
_LOG_LINE = re.compile(r"^(?P<ts>\S+)\s+(?P<level>[A-Z]+)\s+(?P<source>\S+)\s+(?P<event>\S+)\s*(?P<kv>.*)$")


def _parse_log(raw: str) -> list[dict]:
    events = []
    for line in raw.splitlines():
        m = _LOG_LINE.match(line.strip())
        if not m:
            continue
        ev = m.groupdict()
        ev.update(dict(re.findall(r"(\w+)=(\S+)", ev.pop("kv"))))
        events.append(ev)
    return events


def analyze_logs(log: str, username: str | None = None) -> ToolResult:
    """Parse a system log and surface findings (lockout causes, gateway health, security signals)."""
    try:
        events = _parse_log(store.read_log(log))
    except FileNotFoundError:
        return ToolResult("analyze_logs", False, f"No log named '{log}'. Available: vpn, auth.")
    if username:
        events = [e for e in events if e.get("user") in (username, None)]

    levels = Counter(e["level"] for e in events)
    reasons = Counter(e.get("reason") for e in events if e["level"] == "ERROR" and e.get("reason"))
    per_user: dict[str, Counter] = defaultdict(Counter)
    for e in events:
        if e.get("user"):
            per_user[e["user"]][e["event"]] += 1

    findings: list[str] = []
    security_alert = False
    for e in events:
        if e["event"] in ("impossible_travel", "mfa_denied"):
            security_alert = True
            findings.append(f"SECURITY: {e['event']} for user {e.get('user')} from {e.get('new_src') or e.get('src')} at {e['ts']}")
    for user, counts in per_user.items():
        mobile_fails = sum(1 for e in events if e.get("user") == user and e.get("reason") == "bad_password" and "mobile" in e.get("app", ""))
        if counts.get("account_locked") and mobile_fails >= 3:
            findings.append(f"{user}: lockout caused by {mobile_fails} bad-password attempts from a mobile mail app (stale saved password, see KB-002)")
        elif counts.get("account_locked"):
            findings.append(f"{user}: account locked after repeated failed sign-ins")
    unhealthy = [e for e in events if e["event"] == "health_check" and int(e.get("latency_ms", 0)) > int(e.get("threshold_ms", 1e9))]
    if unhealthy:
        gw = Counter(e["source"] for e in unhealthy).most_common(1)[0][0]
        tls = sum(1 for e in events if e["source"] == gw and e["event"] == "tls_handshake_failed")
        findings.append(f"{gw}: {len(unhealthy)} failed health checks (latency up to {max(int(e['latency_ms']) for e in unhealthy)} ms) and {tls} TLS handshake timeouts - gateway degraded")
    for e in events:
        if e.get("reason") == "device_cert_expired":
            findings.append(f"{e.get('user')}: VPN rejected because the device certificate expired (see KB-004)")

    data = {"log": log, "filter_user": username, "events": len(events), "levels": dict(levels),
            "top_error_reasons": reasons.most_common(5), "findings": findings, "security_alert": security_alert}
    msg = f"Analyzed {len(events)} events in {log}.log" + (f" for {username}" if username else "") + ". "
    msg += ("Findings: " + "; ".join(findings)) if findings else "No anomalies found."
    return ToolResult("analyze_logs", True, msg, data,
                      needs_escalation=security_alert,
                      escalation_reason="Security signal detected in logs" if security_alert else "")


def _security_hold(username: str) -> str | None:
    result = analyze_logs("auth", username)
    if result.data.get("security_alert"):
        return next(f for f in result.data["findings"] if f.startswith("SECURITY"))
    return None


# ---------------------------------------------------------------- account automations
def _check_self_service(tool: str, requester: str, target: str | None) -> tuple[dict | None, ToolResult | None]:
    target = (target or requester).lower()
    if target != requester.lower():
        return None, ToolResult(tool, False, f"You can only run {tool.replace('_', ' ')} for your own account. "
                                "Requests for other people need identity verification by a technician.",
                                needs_escalation=True, escalation_reason="Action requested for another user's account")
    user = store.get_user(target)
    if not user:
        return None, ToolResult(tool, False, f"No account named '{target}' in the directory.",
                                needs_escalation=True, escalation_reason="Unknown account")
    if not user["mfa_enrolled"]:
        return None, ToolResult(tool, False, "Your account has no MFA method enrolled, so I can't verify your identity automatically.",
                                needs_escalation=True, escalation_reason="Identity verification requires a technician (no MFA enrolled)")
    hold = _security_hold(target)
    if hold:
        return None, ToolResult(tool, False, "I've paused this request because of unusual sign-in activity on your account. "
                                "A security analyst will contact you.", {"signal": hold},
                                needs_escalation=True, escalation_reason=f"Security hold: {hold}")
    return user, None


def unlock_account(requester: str, username: str | None = None) -> ToolResult:
    """Unlock the requester's own account after MFA verification."""
    user, refusal = _check_self_service("unlock_account", requester, username)
    if refusal:
        return refusal
    cause = analyze_logs("auth", user["username"]).data["findings"]
    if not user["locked"]:
        return ToolResult("unlock_account", True, f"Account {user['username']} is not locked; no change needed.",
                          {"username": user["username"], "was_locked": False, "log_findings": cause})
    store.update_user(user["username"], locked=False)
    return ToolResult("unlock_account", True,
                      f"MFA push approved. Account {user['username']} is unlocked.",
                      {"username": user["username"], "was_locked": True, "mfa_verified": True, "log_findings": cause})


def reset_password(requester: str, username: str | None = None) -> ToolResult:
    """Send a one-time password reset link to the requester's own account after MFA verification."""
    user, refusal = _check_self_service("reset_password", requester, username)
    if refusal:
        return refusal
    was_locked = user["locked"]
    store.update_user(user["username"], locked=False)
    return ToolResult("reset_password", True,
                      f"MFA push approved. A one-time reset link (valid 15 minutes) was sent to {user['email']}."
                      + (" Your account was also unlocked." if was_locked else ""),
                      {"username": user["username"], "mfa_verified": True, "unlocked": was_locked, "link_ttl_min": 15})


# ---------------------------------------------------------------- service status
def check_service_status(service: str | None = None) -> ToolResult:
    """Return live status for one service (vpn, email, sso, wifi, printing, hr_portal, company_portal) or all."""
    services = store.load_services()
    if service:
        match = [s for s in services if s["name"] == service.lower()]
        if not match:
            return ToolResult("check_service_status", False, f"Unknown service '{service}'.",
                              {"known": [s["name"] for s in services]})
        services = match
    lines = [f"{s['display']}: {s['status'].upper()}" + (f" - {s['note']}" if s["note"] else "") for s in services]
    impacted = [s["name"] for s in services if s["status"] != "operational"]
    return ToolResult("check_service_status", True, "\n".join(lines), {"services": services, "impacted": impacted})


# ---------------------------------------------------------------- tickets
PRIORITY_QUEUE = {
    "security": ("P1", "Security Ops"),
    "account_lockout": ("P2", "Service Desk L1"),
    "password_reset": ("P2", "Service Desk L1"),
    "account_access": ("P3", "Identity & Access"),
    "access_request": ("P4", "Identity & Access"),
    "vpn": ("P3", "Network Ops"),
    "network": ("P3", "Network Ops"),
    "email": ("P3", "Service Desk L1"),
    "printer": ("P4", "Desktop Support L2"),
    "hardware": ("P3", "Desktop Support L2"),
    "software": ("P4", "Desktop Support L2"),
    "other": ("P4", "Service Desk L1"),
}


def triage(category: str, urgency: str, requester: str) -> tuple[str, str]:
    """Map category + urgency + VIP status to (priority, queue) per KB-012."""
    priority, queue = PRIORITY_QUEUE.get(category, PRIORITY_QUEUE["other"])
    rank = {"P1": 1, "P2": 2, "P3": 3, "P4": 4}[priority]
    if urgency == "critical":
        rank = 1
    elif urgency == "high":
        rank = min(rank, 2)
    user = store.get_user(requester)
    if user and user.get("vip"):
        rank = min(rank, 2)
    return f"P{rank}", queue


def create_ticket(requester: str, category: str, urgency: str, title: str, summary: str,
                  steps_tried: list[str] | None = None, transcript: str = "") -> ToolResult:
    """Open a ticket routed by the SLA policy; mirrors it to GitHub Issues when configured."""
    priority, queue = triage(category, urgency, requester)
    t = store.create_ticket(requester=requester, category=category, priority=priority, queue=queue,
                            title=title, summary=summary, steps_tried=steps_tried, transcript=transcript)
    url = _mirror_to_github(t)
    if url:
        store.set_external_url(t["id"], url)
        t["external_url"] = url
    return ToolResult("create_ticket", True, f"Ticket {t['number']} opened ({priority}, queue: {queue}).", t)


def log_resolved(requester: str, category: str, title: str, summary: str, resolved_by: str) -> dict:
    """Record a request the assistant resolved itself, for reporting (deflection rate)."""
    return store.create_ticket(requester=requester, category=category, priority="P4", queue="Self-service",
                               title=title, summary=summary, status="resolved", resolved_by=resolved_by)


def _mirror_to_github(ticket: dict) -> str | None:
    cfg = settings()
    if not (cfg.github_token and cfg.github_repo):
        return None
    body = (f"**{ticket['number']}** · {ticket['priority']} · queue: {ticket['queue']}\n\n"
            f"**Requester:** {ticket['requester']}\n\n**Summary:** {ticket['summary']}\n\n"
            f"**Steps already tried:**\n" + "\n".join(f"- {s}" for s in ticket["steps_tried"]) +
            "\n\n_Opened automatically by the Bayview IT Assistant Escalation Agent._")
    payload = {"title": f"[{ticket['priority']}] {ticket['title']}", "body": body,
               "labels": ["helpdesk", ticket["priority"], ticket["category"]]}
    req = urllib.request.Request(
        f"https://api.github.com/repos/{cfg.github_repo}/issues", data=json.dumps(payload).encode(), method="POST",
        headers={"Authorization": f"Bearer {cfg.github_token}", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            return json.loads(resp.read())["html_url"]
    except Exception:
        return None  # GitHub is a mirror; the local ticket is the system of record.


# Registry used by the Workflow Agent and the MCP server.
WORKFLOW_ACTIONS = {
    "reset_password": reset_password,
    "unlock_account": unlock_account,
    "check_service_status": check_service_status,
    "analyze_logs": analyze_logs,
}


In [ ]:
%%writefile helpdesk/agents.py
"""The four agents. Each is a LangGraph node: it reads the shared state and returns a partial update.

    Intake     - redacts PII, classifies the request, extracts entities (Gemini structured output)
    Knowledge  - RAG over the KB in Chroma; answers only from retrieved context, with citations
    Workflow   - runs an automation tool (unlock, reset, status check, log analysis)
    Escalation - writes a technician handoff and opens a routed ticket (optionally a GitHub issue)

Knowledge and Workflow hand off to Escalation by setting ``escalation_reason``.
"""

from __future__ import annotations

import operator
import time
from typing import Annotated, Any, TypedDict

from . import offline, tools
from .config import settings
from .knowledge_base import get_kb
from .llm import get_llm
from .privacy import redact
from .schemas import HandoffSummary, IntakeResult, KnowledgeAnswer

SLA_FIRST_RESPONSE = {"P1": "15 minutes", "P2": "1 hour", "P3": "4 business hours", "P4": "1 business day"}


class HelpdeskState(TypedDict, total=False):
    requester: str
    message: str
    history: list[dict]
    redactions: list[str]
    intake: dict
    retrieved: list[dict]
    tool_result: dict
    answer: str
    sources: list[str]
    escalation_reason: str
    ticket: dict
    resolved_by: str
    trace: Annotated[list[dict], operator.add]  # every agent appends; LangGraph concatenates


def _trace(agent: str, started: float, detail: str, **extra: Any) -> list[dict]:
    return [{"agent": agent, "ms": round((time.perf_counter() - started) * 1000), "detail": detail, **extra}]


# ---------------------------------------------------------------------------- Intake
INTAKE_SYSTEM = """You are the Intake Agent for the Bayview Logistics IT Service Desk.
Classify the employee's message and extract entities. Rules:
- request_type "action" + an action when the user wants their account unlocked, their password reset,
  a system's status checked ("is VPN down?", "is anyone else affected?"), or their sign-in logs investigated
  ("why does my account keep locking?").
- request_type "question" with action "none" for how-to questions, even about passwords ("how do I reset my password?").
- category "security" and request_type "incident" for phishing, clicked links, compromised accounts, unexpected MFA prompts,
  lost/stolen devices, malware. Never pick an automation for security incidents.
- target_username: only when the user asks for an action on a different person's account.
- urgency: critical for security incidents or many users affected; high when the user cannot work; low for how-to.
- wants_human: only when they explicitly ask for a person."""


def intake_agent(state: HelpdeskState) -> dict:
    t0 = time.perf_counter()
    message, redactions = redact(state["message"])
    llm = get_llm()
    if llm.offline:
        result = offline.classify(message, state["requester"])
    else:
        history = "\n".join(f"{h['role']}: {redact(h['content'])[0]}" for h in state.get("history", [])[-6:])
        prompt = (f"Requester username: {state['requester']}\n"
                  + (f"Earlier conversation:\n{history}\n" if history else "")
                  + f"Latest message: {message}")
        result = llm.structured(prompt, IntakeResult, INTAKE_SYSTEM)
    intake = result.model_dump()
    detail = f"{intake['category']} / {intake['request_type']} / action={intake['action']} / urgency={intake['urgency']}"
    return {"message": message, "redactions": redactions, "intake": intake,
            "trace": _trace("Intake", t0, detail, redactions=redactions, confidence=intake["confidence"])}


def route_from_intake(state: HelpdeskState) -> str:
    """Deterministic routing policy on top of the LLM classification (auditable, testable)."""
    i = state["intake"]
    if i["category"] == "security" or i["request_type"] == "incident" or i["wants_human"]:
        return "escalation"
    if i["action"] != "none":
        return "workflow"
    return "knowledge"


# ---------------------------------------------------------------------------- Knowledge
KNOWLEDGE_SYSTEM = """You are the Knowledge Agent for the Bayview Logistics IT Service Desk.
Answer ONLY from the numbered context chunks. Do not invent URLs, settings or policies.
If the context does not answer the question, set grounded=false and say so briefly.
Write for a non-technical employee: a one-line answer, then short numbered steps. Max 150 words.
Cite the chunk ids you used in cited_ids."""


def knowledge_agent(state: HelpdeskState) -> dict:
    t0 = time.perf_counter()
    cfg, llm, kb = settings(), get_llm(), get_kb()
    intake = state["intake"]
    hits = kb.search(state["message"], k=cfg.retrieval_top_k)
    retrieved = [{"id": h.chunk.id, "citation": h.citation, "score": h.score} for h in hits]

    if not hits or hits[0].score < cfg.min_relevance:
        top = hits[0].score if hits else 0
        return {"retrieved": retrieved,
                "escalation_reason": f"No knowledge-base article is relevant enough (best match {top:.2f} < {cfg.min_relevance})",
                "trace": _trace("Knowledge", t0, f"retrieval below threshold (top={top:.2f})", retrieved=retrieved)}

    # Collaboration with the Workflow tools: surface live outages related to the question.
    banner = ""
    if intake.get("service", "none") != "none":
        status = tools.check_service_status(intake["service"])
        if status.ok and status.data["impacted"]:
            banner = f"⚠️ Known issue right now - {status.message}\n\n"

    if llm.offline:
        answer, grounded, cited = offline.extractive_answer(state["message"], hits), True, [hits[0].chunk.id]
    else:
        context = "\n\n".join(f"[{h.chunk.id}] {h.chunk.text}" for h in hits)
        out = llm.structured(f"Context:\n{context}\n\nEmployee question: {state['message']}", KnowledgeAnswer, KNOWLEDGE_SYSTEM)
        answer, grounded, cited = out.answer, out.grounded, out.cited_ids

    if not grounded:
        return {"retrieved": retrieved, "escalation_reason": "Knowledge base does not cover this question",
                "trace": _trace("Knowledge", t0, "model reported answer not grounded in context", retrieved=retrieved)}

    by_id = {h.chunk.id: h for h in hits}
    sources = list(dict.fromkeys(by_id[c].citation for c in cited if c in by_id)) or [hits[0].citation]
    tools.log_resolved(state["requester"], intake["category"], intake["summary"][:80], answer[:500], "Knowledge Agent")
    return {"retrieved": retrieved, "answer": banner + answer, "sources": sources, "resolved_by": "Knowledge Agent",
            "trace": _trace("Knowledge", t0, f"answered from {len(sources)} source(s), top score {hits[0].score:.2f}", retrieved=retrieved)}


# ---------------------------------------------------------------------------- Workflow
WORKFLOW_SYSTEM = """You are the Workflow Agent for the Bayview Logistics IT Service Desk.
An automation just ran. Explain the outcome to the employee in plain language (max 120 words):
what was done, what they should do next, and any tip from the log findings or knowledge snippet.
Never reveal raw log lines, IP addresses, or other users' names."""

_LOG_FOR_SERVICE = {"vpn": "vpn"}


def workflow_agent(state: HelpdeskState) -> dict:
    t0 = time.perf_counter()
    intake, requester = state["intake"], state["requester"]
    action = intake["action"]
    target = intake.get("target_username") or None

    if action in ("reset_password", "unlock_account"):
        result = tools.WORKFLOW_ACTIONS[action](requester, target)
    elif action == "check_service_status":
        svc = intake.get("service")
        result = tools.check_service_status(None if svc in (None, "none") else svc)
    else:  # analyze_logs: scoped to the requester; investigating someone else's logs is a technician task
        result = tools.analyze_logs(_LOG_FOR_SERVICE.get(intake.get("service", ""), "auth"), target or requester)

    # A user-reported outage the status page doesn't know about yet is new information for Network Ops.
    if (result.tool == "check_service_status" and result.ok and not result.data["impacted"]
            and intake.get("service", "none") != "none" and intake["urgency"] in ("high", "critical")):
        result.needs_escalation = True
        result.escalation_reason = f"User reports a {intake['service']} outage that is not on the status page yet"

    detail = f"{result.tool} -> {'ok' if result.ok else 'refused'}"
    if not result.ok or result.needs_escalation:
        return {"tool_result": result.to_dict(),
                "escalation_reason": result.escalation_reason or result.message,
                "trace": _trace("Workflow", t0, detail + " (escalating)", tool=result.to_dict())}

    # Pull one KB tip so the user gets prevention advice, not just the action result.
    hits = get_kb().search(f"{intake['category']} {state['message']}", k=1)
    tip = hits[0] if hits and hits[0].score >= settings().min_relevance else None
    llm = get_llm()
    if llm.offline:
        answer = f"✅ {result.message}"
        findings = [f for f in result.data.get("log_findings", result.data.get("findings", [])) if requester in f]
        if findings:
            answer += "\n\nWhat I found in the sign-in logs: " + "; ".join(f.split(": ", 1)[-1] for f in findings)
        if tip:
            answer += f"\n\nTip from {tip.chunk.doc_id}: " + tip.chunk.text.split("\n", 1)[-1].strip().split("\n")[0]
    else:
        prompt = (f"Employee message: {state['message']}\nAutomation result: {result.to_dict()}\n"
                  + (f"Knowledge snippet [{tip.chunk.id}]: {tip.chunk.text}" if tip else ""))
        answer = llm.text(prompt, WORKFLOW_SYSTEM)

    tools.log_resolved(requester, intake["category"], intake["summary"][:80], result.message, f"Workflow Agent ({result.tool})")
    return {"tool_result": result.to_dict(), "answer": answer, "resolved_by": f"Workflow Agent ({result.tool})",
            "sources": [tip.citation] if tip else [],
            "trace": _trace("Workflow", t0, detail, tool=result.to_dict())}


# ---------------------------------------------------------------------------- Escalation
ESCALATION_SYSTEM = """You are the Escalation Agent for the Bayview Logistics IT Service Desk.
Write a handoff for the human technician so the employee never has to repeat themselves.
Be factual; include what automation or knowledge lookups already happened."""


def escalation_agent(state: HelpdeskState) -> dict:
    t0 = time.perf_counter()
    intake, requester = state["intake"], state["requester"]
    reason = state.get("escalation_reason") or (
        "Security incident - always handled by Security Ops" if intake["category"] == "security"
        else "User asked for a person" if intake["wants_human"] else "Routed to a technician")
    steps = []
    if state.get("retrieved"):
        steps.append("Searched knowledge base: " + ", ".join(r["id"] for r in state["retrieved"][:3]))
    if state.get("tool_result"):
        tr = state["tool_result"]
        steps.append(f"Automation {tr['tool']}: {tr['message']}")
    transcript = "\n".join(f"{h['role']}: {redact(h['content'])[0]}" for h in state.get("history", [])) + f"\nuser: {state['message']}"

    llm = get_llm()
    if llm.offline:
        handoff = HandoffSummary(title=intake["summary"][:80], summary=f"{intake['summary']} Escalation reason: {reason}.",
                                 steps_tried=steps or ["None - routed directly to a technician"])
    else:
        prompt = (f"Requester: {requester}\nClassification: {intake}\nEscalation reason: {reason}\n"
                  f"Steps already taken: {steps}\nTranscript:\n{transcript}")
        handoff = llm.structured(prompt, HandoffSummary, ESCALATION_SYSTEM)

    result = tools.create_ticket(requester, intake["category"], intake["urgency"], handoff.title,
                                 handoff.summary, handoff.steps_tried, transcript)
    t = result.data
    answer = (f"I've passed this to a person. Ticket **{t['number']}** is open with **{t['queue']}** "
              f"(priority {t['priority']}; first response within {SLA_FIRST_RESPONSE[t['priority']]}).\n\n"
              f"Why: {reason}.\n\nThey'll see everything we've covered, so you won't need to repeat yourself.")
    if intake["category"] == "security":
        answer += ("\n\nWhile you wait: don't click any more links, and if you entered your password anywhere, "
                   "change it now from a trusted device (KB-011).")
    if t.get("external_url"):
        answer += f"\n\nTracked on GitHub: {t['external_url']}"
    return {"ticket": t, "answer": answer, "escalation_reason": reason, "resolved_by": "Escalated to human",
            "sources": ["KB-012 · Service Desk priorities, SLAs and escalation"],
            "trace": _trace("Escalation", t0, f"{t['number']} -> {t['queue']} ({t['priority']})", reason=reason)}


def route_to_escalation_if_needed(state: HelpdeskState) -> str:
    return "escalation" if state.get("escalation_reason") and not state.get("answer") else "end"


In [ ]:
%%writefile helpdesk/graph.py
"""LangGraph orchestration.

    START -> intake --(route_from_intake)--> knowledge  --(answered?)--> END
                                       \\--> workflow   --(ok?)-------> END
                                        \\-> escalation <-(otherwise)--/
                                                 |
                                                END
"""

from __future__ import annotations

import time
from functools import lru_cache

from langgraph.graph import END, START, StateGraph

from .agents import (HelpdeskState, escalation_agent, intake_agent, knowledge_agent, route_from_intake,
                     route_to_escalation_if_needed, workflow_agent)
from .config import settings


@lru_cache(maxsize=1)
def build_graph():
    g = StateGraph(HelpdeskState)
    g.add_node("intake", intake_agent)
    g.add_node("knowledge", knowledge_agent)
    g.add_node("workflow", workflow_agent)
    g.add_node("escalation", escalation_agent)

    g.add_edge(START, "intake")
    g.add_conditional_edges("intake", route_from_intake, ["knowledge", "workflow", "escalation"])
    for node in ("knowledge", "workflow"):
        g.add_conditional_edges(node, route_to_escalation_if_needed, {"escalation": "escalation", "end": END})
    g.add_edge("escalation", END)
    return g.compile()


def handle_request(message: str, requester: str, history: list[dict] | None = None) -> dict:
    """Run one user message through the agent graph. Returns the final state plus timing."""
    t0 = time.perf_counter()
    state = build_graph().invoke({"message": message, "requester": requester, "history": history or [], "trace": []})
    state["latency_ms"] = round((time.perf_counter() - t0) * 1000)
    state["path"] = [t["agent"] for t in state["trace"]]
    state["mode"] = settings().mode_label
    return state


def mermaid() -> str:
    """Mermaid source of the compiled graph (used in docs and the notebook)."""
    return build_graph().get_graph().draw_mermaid()


In [ ]:
%%writefile helpdesk/cli.py
"""Command-line demo: python -m helpdesk.cli --user asmith "I'm locked out again"

Without a message it runs a short scripted demo covering every agent path.
"""

from __future__ import annotations

import argparse
import sys

from .config import settings
from .graph import handle_request
from .knowledge_base import get_kb
from .store import reset_runtime

DEMO = [
    ("jdoe", "How do I set up the authenticator app on my new phone?"),
    ("asmith", "I'm locked out of my account again, can you unlock it?"),
    ("pchen", "Is the VPN down? It keeps saying gateway not responding"),
    ("ceo", "I got an MFA prompt I didn't start and my password is Harbor2026!"),
    ("rpatel", "How do I request a standing desk for my office?"),
]


def print_result(user: str, message: str, state: dict) -> None:
    bar = "─" * 78
    print(f"\n{bar}\n👤 {user}: {message}\n{bar}")
    for t in state["trace"]:
        print(f"  ▸ {t['agent']:<10} {t['ms']:>5} ms  {t['detail']}")
    if state.get("redactions"):
        print(f"  🔒 redacted before LLM/ticket: {', '.join(state['redactions'])}")
    print(f"\n🤖 {state['answer']}\n")
    if state.get("sources"):
        print("  📚 Sources: " + " | ".join(state["sources"]))
    print(f"  ⏱  {state['latency_ms']} ms total · path: {' → '.join(state['path'])} · mode: {state['mode']}")


def main(argv: list[str] | None = None) -> int:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("message", nargs="?", help="Message to send (omit to run the scripted demo)")
    p.add_argument("--user", default="jdoe", help="Requester username (see data/directory.json)")
    p.add_argument("--reset", action="store_true", help="Restore seed data and clear tickets first")
    args = p.parse_args(argv)

    if args.reset or not args.message:
        reset_runtime()
    kb = get_kb()
    print(f"Bayview IT Assistant · mode: {settings().mode_label} · KB: {kb.size['articles']} articles / "
          f"{kb.size['chunks']} chunks · embeddings: {kb.llm.embedding_model}")
    for user, msg in ([(args.user, args.message)] if args.message else DEMO):
        print_result(user, msg, handle_request(msg, user))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile app.py
"""Gradio web UI for the Bayview IT Assistant.

    python app.py            # http://127.0.0.1:7860
    python app.py --share    # public link (used in Colab)

Layout: chat on the left; on the right, an "under the hood" panel showing which agents ran,
retrieval scores, the automation result, and the live ticket queue. Showing the agent path
builds user trust and makes the demo explain itself.
"""

from __future__ import annotations

import argparse

import gradio as gr

from helpdesk import store
from helpdesk.config import settings
from helpdesk.graph import handle_request
from helpdesk.knowledge_base import get_kb

AGENT_ICON = {"Intake": "🧭", "Knowledge": "📚", "Workflow": "⚙️", "Escalation": "🙋"}
EXAMPLES = [
    ("jdoe", "How do I set up the authenticator app on my new phone?"),
    ("asmith", "I'm locked out of my account again, can you unlock it?"),
    ("pchen", "Is the VPN down? It keeps saying gateway not responding"),
    ("asmith", "Why does my account keep locking? Can you check the logs?"),
    ("ceo", "I got an MFA prompt I didn't start"),
    ("jdoe", "Please reset the password for asmith"),
]


def _user_choices() -> list[tuple[str, str]]:
    return [(f"{u['name']} ({u['username']}) · {u['department']}", u["username"]) for u in store.list_users()]


def _trace_markdown(state: dict) -> str:
    lines = [f"**Path:** {' → '.join(AGENT_ICON[a] + ' ' + a for a in state['path'])}  ",
             f"**Total:** {state['latency_ms']} ms · **Mode:** `{state['mode']}`", ""]
    for t in state["trace"]:
        lines.append(f"- {AGENT_ICON[t['agent']]} **{t['agent']}** ({t['ms']} ms): {t['detail']}")
    if state.get("redactions"):
        lines.append(f"- 🔒 Redacted before LLM / ticket: {', '.join(state['redactions'])}")
    if state.get("retrieved"):
        lines += ["", "**Retrieved chunks** (cosine similarity)", "", "| Chunk | Score |", "|---|---|"]
        lines += [f"| {r['citation']} | {r['score']:.2f} |" for r in state["retrieved"]]
    if state.get("tool_result"):
        tr = state["tool_result"]
        lines += ["", f"**Automation:** `{tr['tool']}` → {'✅ ok' if tr['ok'] else '⛔ refused'}"]
    if state.get("ticket"):
        t = state["ticket"]
        lines += ["", f"**Ticket:** {t['number']} · {t['priority']} · {t['queue']}", f"> {t['summary']}"]
    return "\n".join(lines)


def _tickets_table() -> list[list]:
    return [[t["number"], t["status"], t["priority"], t["queue"], t["requester"], t["title"][:60]]
            for t in store.list_tickets(limit=15)]


def respond(message: str, history: list[dict], username: str):
    if not message.strip():
        return history, gr.update(), gr.update(), ""
    prior = [{"role": m["role"], "content": m["content"] if isinstance(m["content"], str) else str(m["content"])}
             for m in history]
    state = handle_request(message, username, prior)
    reply = state["answer"]
    if state.get("sources"):
        reply += "\n\n<sub>📚 " + " · ".join(state["sources"]) + "</sub>"
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    return history, _trace_markdown(state), _tickets_table(), ""


def reset_demo():
    store.reset_runtime()
    return [], "_Demo data reset._", _tickets_table()


def build_ui() -> gr.Blocks:
    kb = get_kb()
    with gr.Blocks(title="Bayview IT Assistant") as demo:
        gr.Markdown(
            f"## 🛟 Bayview IT Assistant\n"
            f"Multi-agent IT support · Intake → Knowledge / Workflow / Escalation · "
            f"mode `{settings().mode_label}` · KB {kb.size['articles']} articles / {kb.size['chunks']} chunks")
        with gr.Row():
            with gr.Column(scale=3):
                user = gr.Dropdown(_user_choices(), value="jdoe", label="Signed in as (simulated SSO)")
                chat = gr.Chatbot(height=460, label="Conversation", elem_id="chat",
                                  placeholder="Ask about passwords, VPN, email, printers, software…")
                with gr.Row():
                    box = gr.Textbox(placeholder="Describe your IT issue…", show_label=False, scale=5, autofocus=True, elem_id="msgbox")
                    send = gr.Button("Send", variant="primary", scale=1, elem_id="send")
                gr.Markdown("**Try:**")
                with gr.Row():
                    for who, text in EXAMPLES:
                        btn = gr.Button(f"{who}: {text[:38]}…", size="sm")
                        btn.click(lambda w=who, t=text: (w, t), outputs=[user, box])
            with gr.Column(scale=2):
                gr.Markdown("### 🔍 Under the hood")
                trace = gr.Markdown("_Agent activity appears here after each message._", elem_id="trace")
                tickets = gr.Dataframe(_tickets_table(), headers=["Ticket", "Status", "Pri", "Queue", "User", "Title"],
                                       label="Ticket queue (live)", interactive=False, wrap=True, elem_id="tickets")
                gr.Button("Reset demo data", size="sm").click(reset_demo, outputs=[chat, trace, tickets])
        for trigger in (send.click, box.submit):
            trigger(respond, inputs=[box, chat, user], outputs=[chat, trace, tickets, box])
        user.change(lambda: [], outputs=chat)  # new identity = new conversation
    return demo


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--share", action="store_true")
    ap.add_argument("--port", type=int, default=7860)
    a = ap.parse_args()
    build_ui().launch(share=a.share, server_port=a.port)


In [ ]:
%%writefile mcp_server.py
"""MCP server exposing the helpdesk to any MCP client (VS Code Copilot agent mode, Claude, etc.).

    python mcp_server.py          # stdio transport (what .vscode/mcp.json launches)

Why MCP instead of a REST API: the same tool definitions (name, typed parameters, docstring)
are discovered automatically by every MCP client - no per-client SDK, OpenAPI glue or auth
wrapper to write. A technician in VS Code can ask Copilot "show open P1 tickets and analyze
the auth log for ceo" and the client calls these tools directly; combined with the GitHub MCP
server in the same workspace, escalated issues can be triaged next to the code that fixes them.
"""

from __future__ import annotations

from mcp.server.mcpserver import MCPServer

from helpdesk import store, tools
from helpdesk.graph import handle_request
from helpdesk.knowledge_base import get_kb

server = MCPServer(
    name="bayview-helpdesk",
    instructions="Bayview Logistics IT helpdesk: search the knowledge base, run account automations, "
                 "check service status, analyze logs and manage tickets.",
)


@server.tool()
def ask_it_assistant(message: str, requester: str) -> dict:
    """Send a message through the full multi-agent pipeline as `requester` (a directory username)."""
    s = handle_request(message, requester)
    return {"answer": s["answer"], "path": s["path"], "sources": s.get("sources", []),
            "ticket": s.get("ticket", {}).get("number"), "latency_ms": s["latency_ms"]}


@server.tool()
def search_knowledge_base(query: str, k: int = 4) -> list[dict]:
    """Semantic search over IT knowledge-base articles. Returns chunks with similarity scores."""
    return [{"citation": h.citation, "score": h.score, "text": h.chunk.text} for h in get_kb().search(query, k)]


@server.tool()
def check_service_status(service: str = "") -> dict:
    """Live status for a service (vpn, email, sso, wifi, printing, hr_portal, company_portal); empty = all."""
    return tools.check_service_status(service or None).to_dict()


@server.tool()
def analyze_logs(log: str, username: str = "") -> dict:
    """Analyze the 'vpn' or 'auth' log, optionally filtered to one username; returns findings."""
    return tools.analyze_logs(log, username or None).to_dict()


@server.tool()
def list_tickets(status: str = "", queue: str = "") -> list[dict]:
    """List recent tickets, optionally filtered by status (open/resolved) or queue name."""
    return [{k: t[k] for k in ("number", "status", "priority", "queue", "requester", "title", "created_at")}
            for t in store.list_tickets(status or None, queue or None)]


@server.tool()
def get_ticket(number: str) -> dict:
    """Full ticket detail including summary, steps tried and transcript. number like INC000004."""
    t = store.get_ticket(int(number.upper().removeprefix("INC")))
    return t or {"error": f"{number} not found"}


@server.tool()
def resolve_ticket(number: str, technician: str) -> dict:
    """Mark a ticket resolved by a technician."""
    t = store.update_ticket_status(int(number.upper().removeprefix("INC")), "resolved", technician)
    return t or {"error": f"{number} not found"}


if __name__ == "__main__":
    server.run("stdio")


In [ ]:
%%writefile scripts/mcp_client_demo.py
"""Connect to the helpdesk MCP server over stdio, exactly as VS Code does, and call its tools.

    python scripts/mcp_client_demo.py

Demonstrates the MCP value proposition: the client knows nothing about the helpdesk in advance.
It discovers tool names, descriptions and JSON schemas at runtime, then calls them.
"""

from __future__ import annotations

import asyncio
import json
import sys
from pathlib import Path

from mcp import Client, StdioServerParameters

ROOT = Path(__file__).resolve().parent.parent


def _show(result) -> str:
    if getattr(result, "structured_content", None) is not None:
        payload = result.structured_content
    else:
        payload = "\n".join(getattr(c, "text", str(c)) for c in result.content)
    text = payload if isinstance(payload, str) else json.dumps(payload, indent=2)
    return text if len(text) < 1400 else text[:1400] + "\n…"


async def main() -> None:
    params = StdioServerParameters(command=sys.executable, args=[str(ROOT / "mcp_server.py")], cwd=str(ROOT))
    async with Client(params) as client:
        info = client.server_info
        print(f"Connected to MCP server: {info.name if info else 'bayview-helpdesk'}\n")
        tools = (await client.list_tools()).tools
        print(f"Discovered {len(tools)} tools:")
        for t in tools:
            params_ = ", ".join(t.input_schema.get("properties", {}).keys())
            print(f"  • {t.name}({params_}) - {(t.description or '').splitlines()[0]}")

        calls = [
            ("check_service_status", {"service": "vpn"}),
            ("analyze_logs", {"log": "auth", "username": "asmith"}),
            ("ask_it_assistant", {"message": "I clicked a link in a weird invoice email", "requester": "jdoe"}),
            ("list_tickets", {"status": "open"}),
        ]
        for name, args in calls:
            print(f"\n▶ call_tool {name} {json.dumps(args)}")
            print(_show(await client.call_tool(name, args)))


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    asyncio.run(main())


In [ ]:
%%writefile eval/run_eval.py
"""Evaluation harness: runs every scenario in eval/scenarios.json through the agent graph.

    python eval/run_eval.py                 # all scenarios
    python eval/run_eval.py --judge         # + LLM-as-judge satisfaction scores (Gemini mode only)
    python eval/run_eval.py --pace 4        # seconds between Gemini calls (free-tier rate limits)

Metrics
  - Intake category accuracy and action accuracy
  - Routing accuracy: did the right agent produce the final answer?
  - Escalation precision / recall: escalate when (and only when) a human is needed
  - Retrieval hit@1 / hit@k for questions with a known source article
  - Containment rate: share of requests resolved without a human
  - Latency p50 / p95 / max
  - Satisfaction (1-5): LLM judge rating helpfulness, accuracy and tone of each answer

Outputs go to eval/results/<mode>/: results.csv, summary.json, report.md, charts.png
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import statistics
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

from pydantic import BaseModel, Field  # noqa: E402

from helpdesk.config import reset_settings, settings  # noqa: E402
from helpdesk.graph import handle_request  # noqa: E402
from helpdesk.knowledge_base import get_kb  # noqa: E402
from helpdesk.llm import get_llm  # noqa: E402
from helpdesk.store import reset_runtime  # noqa: E402

FINAL = {"Knowledge": "knowledge", "Workflow": "workflow", "Escalation": "escalation"}


class JudgeScore(BaseModel):
    helpfulness: int = Field(description="1-5: does the reply move the user toward resolution?")
    accuracy: int = Field(description="1-5: is it correct and consistent with the expected behaviour?")
    tone: int = Field(description="1-5: clear, friendly, non-technical where appropriate?")
    rationale: str = Field(description="One sentence.")


JUDGE_SYSTEM = """You evaluate an IT helpdesk assistant's reply as if you were the employee who asked.
Score 1-5 for helpfulness, accuracy and tone. An appropriate escalation to a human (e.g. security
incidents, actions on someone else's account, out-of-scope requests) deserves high scores if it is
clearly explained."""


def percentile(values: list[float], p: float) -> float:
    s = sorted(values)
    k = (len(s) - 1) * p
    lo, hi = int(k), min(int(k) + 1, len(s) - 1)
    return round(s[lo] + (s[hi] - s[lo]) * (k - lo), 1)


def run(args) -> dict:
    scenarios = json.loads((ROOT / "eval" / "scenarios.json").read_text(encoding="utf-8"))["scenarios"]
    if args.only:
        scenarios = [s for s in scenarios if s["id"].startswith(tuple(args.only.split(",")))]
    cfg, kb, llm = settings(), get_kb(), get_llm()
    mode = "offline" if cfg.offline else "gemini"
    out_dir = ROOT / "eval" / "results" / mode
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Evaluating {len(scenarios)} scenarios · mode={cfg.mode_label} · embeddings={llm.embedding_model}")

    rows = []
    for s in scenarios:
        reset_runtime()  # every scenario starts from the same directory/ticket state
        t0 = time.perf_counter()
        try:
            state = handle_request(s["message"], s["requester"])
            error = ""
        except Exception as e:  # keep going; a crash counts as a failure for that scenario
            state, error = {"intake": {}, "path": [], "answer": "", "latency_ms": round((time.perf_counter() - t0) * 1000)}, repr(e)[:200]
        intake = state.get("intake") or {}
        final = FINAL.get(state["path"][-1], "") if state["path"] else ""
        hits = kb.search(s["message"], k=cfg.retrieval_top_k) if s.get("expected_doc") else []
        ranked_docs = [h.chunk.doc_id for h in hits]
        row = {
            "id": s["id"], "requester": s["requester"], "message": s["message"],
            "expected_category": s["expected_category"] or "*", "category": intake.get("category", ""),
            "category_ok": s["expected_category"] is None or intake.get("category") == s["expected_category"],
            "expected_action": s["expected_action"], "action": intake.get("action", ""),
            "action_ok": intake.get("action") == s["expected_action"],
            "expected_final": s["expected_final"], "final": final, "route_ok": final == s["expected_final"],
            "path": " > ".join(state["path"]),
            "expected_doc": s.get("expected_doc") or "",
            "hit_at_1": bool(ranked_docs) and ranked_docs[0] == s.get("expected_doc"),
            "hit_at_k": s.get("expected_doc") in ranked_docs if s.get("expected_doc") else "",
            "top_score": hits[0].score if hits else "",
            "latency_ms": state["latency_ms"], "answer": state.get("answer", ""), "error": error,
        }
        if args.judge and not llm.offline and not error:
            try:
                expected = f"expected final agent: {s['expected_final']}, expected action: {s['expected_action']}"
                j = llm.structured(f"Employee ({s['requester']}): {s['message']}\nExpected behaviour: {expected}\n"
                                   f"Assistant reply:\n{row['answer']}", JudgeScore, JUDGE_SYSTEM)
                row.update(sat_helpfulness=j.helpfulness, sat_accuracy=j.accuracy, sat_tone=j.tone,
                           satisfaction=round((j.helpfulness + j.accuracy + j.tone) / 3, 2), judge_note=j.rationale)
            except Exception as e:
                row["judge_note"] = f"judge failed: {e!r}"[:120]
        mark = "✅" if row["route_ok"] and row["category_ok"] else "❌"
        print(f"  {mark} {s['id']:<4} {row['path']:<32} {row['latency_ms']:>6} ms  {s['message'][:58]}")
        rows.append(row)

    summary = summarize(rows, mode, cfg, llm)
    write_outputs(rows, summary, out_dir)
    return summary


def summarize(rows: list[dict], mode: str, cfg, llm) -> dict:
    n = len(rows)
    lat = [r["latency_ms"] for r in rows]
    retrieval = [r for r in rows if r["expected_doc"]]
    should = [r for r in rows if r["expected_final"] == "escalation"]
    did = [r for r in rows if r["final"] == "escalation"]
    tp = sum(1 for r in did if r["expected_final"] == "escalation")
    sat = [r["satisfaction"] for r in rows if "satisfaction" in r]
    pct = lambda k: round(100 * sum(1 for r in rows if r[k]) / n, 1)  # noqa: E731
    return {
        "mode": cfg.mode_label, "embedding_model": llm.embedding_model, "scenarios": n,
        "category_accuracy_pct": pct("category_ok"), "action_accuracy_pct": pct("action_ok"),
        "routing_accuracy_pct": pct("route_ok"),
        "end_to_end_pass_pct": round(100 * sum(1 for r in rows if r["route_ok"] and r["category_ok"] and r["action_ok"]) / n, 1),
        "retrieval_hit_at_1_pct": round(100 * sum(1 for r in retrieval if r["hit_at_1"]) / len(retrieval), 1) if retrieval else None,
        "retrieval_hit_at_k_pct": round(100 * sum(1 for r in retrieval if r["hit_at_k"]) / len(retrieval), 1) if retrieval else None,
        "retrieval_k": cfg.retrieval_top_k,
        "escalation_precision_pct": round(100 * tp / len(did), 1) if did else None,
        "escalation_recall_pct": round(100 * tp / len(should), 1) if should else None,
        "containment_rate_pct": round(100 * sum(1 for r in rows if r["final"] in ("knowledge", "workflow")) / n, 1),
        "latency_p50_ms": percentile(lat, 0.5), "latency_p95_ms": percentile(lat, 0.95), "latency_max_ms": max(lat),
        "satisfaction_avg_1to5": round(statistics.mean(sat), 2) if sat else None,
        "llm_calls": llm.stats.llm_calls, "llm_retries": llm.stats.retries, "errors": sum(1 for r in rows if r["error"]),
        "min_relevance": cfg.min_relevance,
    }


def write_outputs(rows: list[dict], summary: dict, out_dir: Path) -> None:
    fields = list(dict.fromkeys(k for r in rows for k in r))
    with open(out_dir / "results.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    labels = {
        "category_accuracy_pct": "Intake category accuracy", "action_accuracy_pct": "Action selection accuracy",
        "routing_accuracy_pct": "Routing accuracy (final agent)", "end_to_end_pass_pct": "End-to-end pass rate",
        "retrieval_hit_at_1_pct": "Retrieval hit@1", "retrieval_hit_at_k_pct": f"Retrieval hit@{summary['retrieval_k']}",
        "escalation_precision_pct": "Escalation precision", "escalation_recall_pct": "Escalation recall",
        "containment_rate_pct": "Containment (no human needed)",
    }
    md = [f"# Evaluation report - {summary['mode']}", "",
          f"{summary['scenarios']} scenarios · embeddings `{summary['embedding_model']}` · "
          f"relevance threshold {summary['min_relevance']} · {summary['llm_calls']} LLM calls · {summary['errors']} errors", "",
          "| Metric | Value |", "|---|---|"]
    md += [f"| {label} | {summary[k]}% |" for k, label in labels.items() if summary[k] is not None]
    md += [f"| Latency p50 / p95 / max | {summary['latency_p50_ms']} / {summary['latency_p95_ms']} / {summary['latency_max_ms']} ms |"]
    if summary["satisfaction_avg_1to5"] is not None:
        md.append(f"| Satisfaction (LLM judge, 1-5) | {summary['satisfaction_avg_1to5']} |")
    md += ["", "## Failures", ""]
    fails = [r for r in rows if not (r["route_ok"] and r["category_ok"] and r["action_ok"])]
    if not fails:
        md.append("None.")
    else:
        md += ["| ID | Message | Expected | Got |", "|---|---|---|---|"]
        md += [f"| {r['id']} | {r['message']} | {r['expected_category']} / {r['expected_action']} / {r['expected_final']} | "
               f"{r['category']} / {r['action']} / {r['final']} |" for r in fails]
    (out_dir / "report.md").write_text("\n".join(md) + "\n", encoding="utf-8")
    _charts(rows, summary, labels, out_dir / "charts.png")
    print(f"\nWrote {out_dir.relative_to(ROOT)}/ (results.csv, summary.json, report.md, charts.png)")


def _charts(rows, summary, labels, path: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1.4, 1, 1]})
    keys = [k for k in labels if summary[k] is not None]
    vals = [summary[k] for k in keys]
    ax = axes[0]
    bars = ax.barh([labels[k] for k in keys][::-1], vals[::-1], color="#2a6fdb")
    ax.bar_label(bars, fmt="%.0f%%", padding=3)
    ax.set_xlim(0, 110)
    ax.set_title("Quality metrics")

    ax = axes[1]
    ax.hist([r["latency_ms"] for r in rows], bins=12, color="#1f9d74")
    ax.axvline(summary["latency_p95_ms"], color="#c0392b", ls="--", label=f"p95 {summary['latency_p95_ms']:.0f} ms")
    ax.set_title("End-to-end latency")
    ax.set_xlabel("ms")
    ax.legend()

    ax = axes[2]
    order = ["knowledge", "workflow", "escalation"]
    grid = [[sum(1 for r in rows if r["expected_final"] == e and r["final"] == g) for g in order] for e in order]
    ax.imshow(grid, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, grid[i][j], ha="center", va="center", color="white" if grid[i][j] > 6 else "black", fontsize=13)
    ax.set_xticks(range(3), order)
    ax.set_yticks(range(3), order)
    ax.set_xlabel("actual final agent")
    ax.set_ylabel("expected")
    ax.set_title("Routing confusion matrix")
    fig.suptitle(f"Bayview IT Assistant evaluation · {summary['mode']} · {summary['scenarios']} scenarios")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    ap = argparse.ArgumentParser()
    ap.add_argument("--judge", action="store_true", help="LLM-as-judge satisfaction scoring (Gemini only)")
    ap.add_argument("--pace", type=float, default=None, help="Seconds between Gemini calls")
    ap.add_argument("--only", default="", help="Comma-separated scenario id prefixes, e.g. K,S0")
    a = ap.parse_args()
    if a.pace is not None:
        os.environ["GEMINI_MIN_INTERVAL"] = str(a.pace)
        reset_settings()
    s = run(a)
    print(json.dumps(s, indent=2))


In [ ]:
import sys, importlib
sys.path.insert(0, '.')
for m in [m for m in list(sys.modules) if m.startswith('helpdesk')]:
    del sys.modules[m]  # pick up rewritten files on re-run
from helpdesk.config import settings
print('Mode:', settings().mode_label)

## 3 · Knowledge base and RAG
Articles are split by section, embedded with Gemini embeddings and stored in Chroma (cosine).

In [ ]:
from helpdesk.knowledge_base import get_kb
kb = get_kb()
print(kb.size, '· embedding model:', kb.llm.embedding_model)
for h in kb.search('VPN says certificate error', k=3):
    print(f'{h.score:.3f}  {h.citation}')

## 4 · The agent graph (LangGraph)

In [ ]:
from helpdesk.graph import build_graph, mermaid
from IPython.display import Image, Markdown, display
try:
    display(Image(build_graph().get_graph().draw_mermaid_png()))
except Exception:
    display(Markdown('```mermaid\n' + mermaid() + '\n```'))

## 5 · Scripted demo - every agent path
Knowledge answer · workflow automation · status check · security escalation (with PII redaction) · out-of-scope escalation.

In [ ]:
!python -m helpdesk.cli

### Try your own message

In [ ]:
from helpdesk.graph import handle_request
from helpdesk.cli import print_result
msg, user = 'Why does my account keep locking? Can you check the logs?', 'asmith'
print_result(user, msg, handle_request(msg, user))

## 6 · MCP integration
The helpdesk is also an **MCP server** (`mcp_server.py`). This client discovers its tools at runtime over stdio - exactly what VS Code does via `.vscode/mcp.json`, alongside the GitHub MCP server.

In [ ]:
!python scripts/mcp_client_demo.py

## 7 · Web UI (Gradio)
Opens a public `*.gradio.live` link. The right panel shows which agents ran, retrieval scores and the live ticket queue.

In [ ]:
from app import build_ui
build_ui().launch(share=True, debug=False)